# PKG customer metrics v3: one row per customer per month

**PNC Treasury Management · Data Science · Payment Knowledge Graph**

Rebuild of `pkg_customer_metrics_v2.ipynb` on the October 2026 counterparty work: entity map v2, canonical bank names,
and the counterparty metrics table. Same grain, same identity columns and the same column names for everything v2 already
had, so attrition and fraud code can switch tables without renaming features.

**Outputs** (database `bdahd01p_dlcdi1_cdi_tm`, external parquet, partitioned by `month`)

| Table | Grain | Purpose |
|---|---|---|
| `pkg_cust_metrics_monthly_v3` | one row per customer per month | identity columns, then every metric |
| `pkg_cust_pair_month_v3` | customer × direction × scope × counterparty entity | intermediate, reusable (adds the pair's canonical bank) |
| `pkg_cust_fi_month` | TM customer × direction × canonical bank | competitor share, banks holding own accounts, which bank was lost and why |
| `pkg_cust_metrics_dictionary_v3` | one row per column of the three tables | definition, type, empty-value rule, version added, what changed in v3 |

v2 tables are left untouched, so the attrition app and the fraud module keep working until they are pointed at `_v3`.

**What changes from v2**

1. **Counterparties are counted at `ent_l2` of `pkg_cpty_entity_map_v2`** (v2: `ent_l1` of map v1). The pair key is the
   `ent_id` of `pkg_cpty_metrics_monthly`, so the two tables join directly. It is also the only level where a counterparty
   that changes banks stays the same counterparty: the routing number is part of the raw id.
2. **Canonical bank names everywhere.** Each counterparty id gets its bank for the month from `pkg_cpty_month` (largest
   amount), mapped through `pkg_fi_routing` (routing prefix of the id) or `pkg_fi_name_map`. Bank metrics are computed at
   id level, not from the pair's first bank as in v2. Mergers (Comerica to Fifth Third and the rest) no longer show as a
   lost bank plus a new one.
3. **Bank changes driven by the counterparty.** A lost or broken bank is flagged when at least `MOVE_SHARE` of the dollars
   the customer last had at that bank now flow with the same counterparties at other banks (the ClickPay case). New
   columns sit next to the v2 ones: `lost_fi_cpty_moved`, `fi_rec_broken_cpty_moved`, `fi_rec_broken_net`,
   `cpty_bank_moved`.
4. **Hub exposure and links to the counterparty table:** share of external dollars and number of counterparties at each
   `hub_tier`, and with eligible counterparties.
5. **Banks:** largest bank by name (`top_fi`), share at PNC as holding bank, largest bank holding own-name accounts
   (`selfpay_top_fi`), and the long table `pkg_cust_fi_month`.
6. **Name classes from typer v2** for the business / person shares.
7. **Fully dark customers keep a row** while any standing relationship or standing bank is broken (v2 kept a row only
   for the month right after the last activity).
8. **Resumable:** a month is done only with its marker for the current configuration, its inputs and the previous
   month's build. Steps are saved to disk, each output is staged, reconciled exactly to the edges, then moved into place.

**Conventions (unchanged from v2)**
- Column names `{metric}_{scope}_{dir}`. Scope: `int` = another PNC customer, `ext` = an external counterparty, `self` =
  the same `mdm_id` on both sides, `all` = `int` + `ext`. Direction is the **customer's** view: `out` = paid,
  `in` = received (the counterparty table uses the opposite view).
- Customer attributes are current-state: `pkg_cust_dim` (latest `load_dt` per `mdm_id`) applied to every month.
- Empty values: amounts and counts are 0 when there is no activity; shares are NULL when the denominator is 0;
  names are NULL when there is nothing to name.

**How to run.** Run all. The build is sequential from 2024-01 because churn needs the previous month. Months already
done are skipped. A stop at any point resumes where it left off: rerun the run cell. A change to a setting that changes
the numbers, or to the map, bank tables, `pkg_cpty_dim` or `pkg_cust_dim`, rebuilds from 2024-01, so settle the inputs
first. For a trial, set `MAX_MONTHS_PER_RUN = 1` and look at the minutes and sizes in section 5.

In [ ]:
# §0a  Setup. Compat shims first: Cloudera SPARK3 PySpark 3.3 predates NumPy 1.24/2.0 and pandas 2.0
import numpy as np, pandas as pd, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
for _alias, _target in {"bool": np.bool_, "object": np.object_}.items():
    if _alias not in np.__dict__:
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):          # removed in pandas 2.0; PySpark 3.3 createDataFrame still calls it
    pd.DataFrame.iteritems = pd.DataFrame.items
    pd.Series.iteritems = pd.Series.items
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")
warnings.filterwarnings("ignore", category=FutureWarning)

import datetime as dt, hashlib, html as _html, json, time, uuid
from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

spark = (SparkSession.builder
         .appName("pkg_customer_metrics_v3")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g")
         .config("spark.executor.cores", "2")
         .config("spark.executor.memoryOverhead", "3g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.dynamicAllocation.enabled", "true")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
         .enableHiveSupport()
         .getOrCreate())
MD = StorageLevel.MEMORY_AND_DISK
print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# §0b  Config
DB              = "bdahd01p_dlcdi1_cdi_tm"
CUST_TABLE      = "dsihd01p_dsi.neo4j_customer"
METRICS_VERSION = "cust-v3.0"          # part of the signature: bump only when the numbers change
OUT_SUFFIX      = "_v3"                # v3 tables sit next to v2; "" would replace v2 in place (not recommended)

HISTORY_START   = "2024-01"            # first month of the edge tables: churn state starts empty here
LAST_MONTH      = None                 # None = latest month present in every monthly input
MAX_MONTHS_PER_RUN = None              # e.g. 1 for a first trial, 8 to build in chunks
REBUILD_FROM    = None                 # e.g. "2026-08": rebuild that month and every later one. Needs the previous
                                       # month's state (only the last KEEP_STATES are kept), else HISTORY_START.
                                       # Set back to None after the run
BUILD_CUST_DIM  = False                # True rebuilds pkg_cust_dim from neo4j_customer: new fingerprint, full rebuild

LOOKBACK        = 12                   # new = not seen in the previous LOOKBACK months (≤ 30)
REC_WINDOW, REC_MIN = 6, 4             # standing = present in ≥ REC_MIN of the previous REC_WINDOW months
KEEP_STATES     = 2                    # churn-state months kept on disk (older ones pruned)
MOVE_SHARE      = 0.5                  # lost/broken bank is counterparty-driven when ≥ this share of its last dollars
                                       # now flows with the same counterparties at other banks
MIN_NAICS_CP    = 5                    # sector metrics need ≥ this many internal counterparties with a valid NAICS
HUB_TIERS       = [1, 2, 3]            # hub_tier ≥ k of pkg_cpty_metrics_monthly (P99 / P99.9 / P99.99)
PNC_FI          = ["PNC BANK"]         # canonical names of PNC as holding bank (not a competitor); checked in §1
CUST_FI_TM_ONLY = True                 # pkg_cust_fi_month for TM customers only (False: every customer, ~6× larger)
MAP_MISS_MAX    = 0                    # counterparty ids of a month allowed to be missing from map v2
CPM_MISS_MAX_SHARE = 0.0001            # share of external dollars allowed with no row in pkg_cpty_metrics_monthly
PREFLIGHT       = True                 # §1b: monthly totals and the largest flows of the latest months
PREFLIGHT_MONTHS = 6
COMPARE_V2      = True                 # §5: v2 vs v3 on the latest month built by both

RAILS = ["ACH", "WIRE", "RTP_PRT", "RTP_P2P", "CHECK", "PCARD"]
RAIL_OF_DIGIT = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP_PRT", "6": "RTP_P2P",
                 "7": "CHECK", "8": "DEBIT_CARD", "9": "PCARD"}
NAICS_PLACEHOLDERS = ["******", "-1", "0", "00", "000000", "999999", "UNKNOWN", "N/A", "NA", "NULL", "NONE"]
LEGAL = r"\b(N A|INC|INCORPORATED|LLC|L L C|CO|CORP|CORPORATION|COMPANY|LTD|LIMITED|LP|LLP|PC|PLLC|PA|NA|THE)\b"
FILES = {"pairs": 8, "metrics": 6, "cust_fi": 2}
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))

# ---- tables
T_EDGE_M, T_EDGE_D = f"{DB}.pkg_edge_monthly", f"{DB}.pkg_edge_daily"
T_CPTY_MONTH, T_CPTY_DIM = f"{DB}.pkg_cpty_month", f"{DB}.pkg_cpty_dim"
T_MAP = f"{DB}.pkg_cpty_entity_map_v2"
T_FI_ROUTING, T_FI_NAMEMAP = f"{DB}.pkg_fi_routing", f"{DB}.pkg_fi_name_map"
T_CPM = f"{DB}.pkg_cpty_metrics_monthly"
T_CUST_DIM = f"{DB}.pkg_cust_dim"
T_V2_METRICS = f"{DB}.pkg_cust_metrics_monthly"
OUT = {"pairs":   f"{DB}.pkg_cust_pair_month{OUT_SUFFIX}",
       "metrics": f"{DB}.pkg_cust_metrics_monthly{OUT_SUFFIX}",
       "cust_fi": f"{DB}.pkg_cust_fi_month",
       "dict":    f"{DB}.pkg_cust_metrics_dictionary{OUT_SUFFIX}"}

In [ ]:
# §0c  Helpers: file system, markers, fingerprints, display
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))

def rm(p):
    if exists(p):
        fs_for(p).delete(JPath(p), True)

def mv(src, dst):
    """Move a finished staging folder into place (the destination is replaced)."""
    rm(dst)
    fs = fs_for(src)
    fs.mkdirs(JPath(dst).getParent())
    if not fs.rename(JPath(src), JPath(dst)):
        raise RuntimeError(f"rename failed: {src} -> {dst}")

def write_json(p, obj):
    out = fs_for(p).create(JPath(p), True)
    out.write(bytearray(json.dumps(obj, sort_keys=True, default=str).encode("utf-8")))
    out.close()

def read_json(p):
    if not exists(p):
        return None
    st = fs_for(p).open(JPath(p))
    rd = spark._jvm.java.io.BufferedReader(spark._jvm.java.io.InputStreamReader(st, "UTF-8"))
    lines, line = [], rd.readLine()
    while line is not None:
        lines.append(line); line = rd.readLine()
    rd.close()
    try:
        return json.loads("\n".join(lines))
    except ValueError:
        return None

def list_months(base):
    if not exists(base):
        return []
    return sorted(s.getPath().getName().split("=", 1)[1] for s in fs_for(base).listStatus(JPath(base))
                  if s.isDirectory() and s.getPath().getName().startswith("month="))

def path_fp(p):
    """Fingerprint of a folder: data files, bytes, latest modification (files and folders starting with _ or . skipped)."""
    if not exists(p):
        return "missing"
    it, n, size, mt = fs_for(p).listFiles(JPath(p), True), 0, 0, 0
    base = str(fs_for(p).makeQualified(JPath(p)))
    while it.hasNext():
        st = it.next()
        rel = str(st.getPath())[len(base):]
        if any(part.startswith(("_", ".")) for part in rel.split("/") if part):
            continue
        n, size, mt = n + 1, size + st.getLen(), max(mt, st.getModificationTime())
    return f"{n}:{size}:{mt}"

def table_exists(t):
    try:
        spark.table(t).schema
        return True
    except Exception:
        return False

def table_location(t):
    try:
        for r in spark.sql(f"DESCRIBE FORMATTED {t}").collect():
            if str(r[0]).strip().lower() == "location":
                return str(r[1]).strip().rstrip("/")
    except Exception:
        pass
    return f"{LOC}/{t.split('.', 1)[1]}"

def prev_month(m):
    y, mo = map(int, m.split("-"))
    return f"{y - 1:04d}-12" if mo == 1 else f"{y:04d}-{mo - 1:02d}"

def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out

def months_between(a, b):
    ya, ma = map(int, a.split("-")); yb, mb = map(int, b.split("-"))
    return (yb - ya) * 12 + (mb - ma)

def md5(obj):
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

def name_key(c):
    """Upper-case, letters/digits/spaces only, legal suffixes removed. Same function on both sides of self-pay matching."""
    s = F.regexp_replace(F.upper(c), "[^A-Z0-9 ]", " ")
    s = F.trim(F.regexp_replace(F.regexp_replace(s, LEGAL, " "), r"\s+", " "))
    return F.when(F.length(s) >= 3, s)

# ---- display: one consolidated HTML output per cell -------------------------------------------------------------------
_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin:2px 0 10px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}"
        ".pkgt td:first-child,.pkgt th:first-child{text-align:left}"
        ".pkgh{margin:12px 0 3px;font-size:13px;font-weight:600}.pkgn{color:#6b6a66;margin:1px 0 6px;font-size:12px}"
        ".pkgok{color:#1e7b34;font-weight:600}.pkgbad{color:#b3261e;font-weight:600}</style>")

def _num(s):
    if s.dtype == object:
        try:
            return pd.to_numeric(s)
        except (ValueError, TypeError):
            return s
    return s

def fmt_df(df, pct=()):
    out = pd.DataFrame(index=df.index)
    for c in df.columns:
        s = _num(df[c])
        if c in pct:
            out[c] = [("—" if pd.isna(v) else f"{v:.2%}") for v in s]
        elif pd.api.types.is_bool_dtype(s):
            out[c] = s.astype(str)
        elif pd.api.types.is_numeric_dtype(s):
            whole = bool((s.dropna() % 1 == 0).all())
            out[c] = [("—" if pd.isna(v) else (f"{int(v):,}" if whole else f"{v:,.3f}")) for v in s]
        else:
            out[c] = s.where(s.notna(), "—").astype(str)
    return out

def H(text):  return ("h", text)
def N(text):  return ("n", text)
def OK(text): return ("ok", text)
def BAD(text): return ("bad", text)
def TB(title, df, pct=()): return ("t", title, df, pct)

def render(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        k = b[0]
        if k in ("h", "n", "ok", "bad"):
            cls = {"h": "pkgh", "n": "pkgn", "ok": "pkgok", "bad": "pkgbad"}[k]
            parts.append(f"<div class='{cls}'>{_html.escape(str(b[1]))}</div>")
        else:
            _, title, df, pct = b
            parts.append(f"<div class='pkgh'>{_html.escape(title)} <span class='pkgn'>[{len(df):,} rows]</span></div>")
            parts.append(fmt_df(df, pct).to_html(index=False, escape=True, border=0, classes="pkgt"))
    return HTML(_CSS + "<div style=\"font-family:'IBM Plex Sans',Arial,sans-serif\">" + "".join(parts) + "</div>")

def show(*blocks):
    display(render(*blocks))

# ---- locations ---------------------------------------------------------------------------------------------------------
LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if str(r[0]).lower() == "location"][0].rstrip("/")
OUT_P = {k: f"{LOC}/{t.split('.', 1)[1]}" for k, t in OUT.items()}
WORK  = f"{LOC}/_tmp_pkg_cust_metrics{OUT_SUFFIX or '_v3'}"         # per-month steps, deleted when the month is published
STATE = f"{LOC}/pkg_cust_churn_state{OUT_SUFFIX or '_v3'}"          # churn state (working, not a Hive table)
STATIC = f"{WORK}/_static"
state_path = lambda kind, m: f"{STATE}/kind={kind}/month={m}"
work_path  = lambda m, step: f"{WORK}/month={m}/{step}"
stage_path = lambda key, m: f"{OUT_P[key]}/_stage_month={m}"
month_dir  = lambda key, m: f"{OUT_P[key]}/month={m}"
MARKER = "_PKG_MONTH.json"
print("database location:", LOC)

## §1 Inputs and lookups

- **Inputs exist and join keys are strings** (the 2026-07 incident was an int64/str key that typed every counterparty
  `unknown` and still produced plausible output).
- **Every lookup is one row per key**, asserted after a deterministic reduction:
  map v2 keeps the lowest `ent_l2` per `cpty_id` (the known duplicate ids), `pkg_cpty_dim` keeps the largest-amount row,
  the bank tables must already be unique.
- `pkg_cust_dim` is reused from v2 unless `BUILD_CUST_DIM = True`.
- The **signature** covers every setting that changes the numbers and a fingerprint of map v2, both bank tables,
  `pkg_cpty_dim` and `pkg_cust_dim`. A new signature makes every month stale.
- The static lookups are written once per signature under the work folder.

In [ ]:
# §1a  Customer dimension (pkg_cust_dim, same build as v2) and input checks
CUST_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
             "customer_start_dt", "naics_cd", "naics_desc", "load_dt"]
IN_LOC = {k: table_location(t) for k, t in {"edge_m": T_EDGE_M, "edge_d": T_EDGE_D, "cpty_month": T_CPTY_MONTH,
                                             "cpm": T_CPM}.items()}

def build_cust_dim():
    EM_all = spark.read.parquet(IN_LOC["edge_m"])
    raw = (spark.table(CUST_TABLE).select(*[nz(c).alias(c) for c in CUST_COLS])
                .filter(F.col("mdm_id").isNotNull()).withColumn("party_type", F.upper("party_type")))
    w = Window.partitionBy("mdm_id").orderBy(F.col("load_dt").desc_nulls_last(), F.col("customer_pwr_id").isNotNull().desc(),
                                             F.col("customer_pwr_id").asc_nulls_last())
    latest = (raw.withColumn("n_load_versions", F.count("*").over(Window.partitionBy("mdm_id")))
                 .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))
    nc = F.upper(F.col("naics_cd"))
    latest = (latest
        .withColumn("naics_status", F.when(nc.isNull(), "missing")
                                     .when(nc.isin(NAICS_PLACEHOLDERS) | ~nc.rlike(r"^[0-9]{2}"), "placeholder").otherwise("valid"))
        .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring("naics_cd", 1, 2)))
        .withColumn("is_tm", F.col("customer_pwr_id").isNotNull()))
    fam = latest.filter(F.col("master_pwr_id").isNotNull()).groupBy("master_pwr_id").agg(F.count("*").alias("family_size"))
    shared = (latest.filter(F.col("customer_pwr_id").isNotNull()).groupBy("customer_pwr_id")
                    .agg(F.count("*").alias("_n")).filter("_n > 1").select("customer_pwr_id", F.lit(True).alias("pwr_shared")))
    latest = (latest.join(fam, "master_pwr_id", "left").join(shared, "customer_pwr_id", "left")
                    .withColumn("in_neo4j_customer", F.lit(True)))
    seen = (EM_all.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("mdm_id"), "month")
              .unionByName(EM_all.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("mdm_id"), "month"))
              .groupBy("mdm_id").agg(F.min("month").alias("first_edge_month"), F.max("month").alias("last_edge_month")))
    dim = (latest.join(seen, "mdm_id", "full")
             .withColumn("in_neo4j_customer", F.coalesce("in_neo4j_customer", F.lit(False)))
             .withColumn("is_tm", F.coalesce("is_tm", F.lit(False)))
             .withColumn("pwr_shared", F.coalesce("pwr_shared", F.lit(False)))
             .withColumn("naics_status", F.coalesce("naics_status", F.lit("missing")))
             .select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                     "naics_cd", "naics_desc", "naics2", "naics_status", "is_tm", "pwr_shared", "family_size",
                     "customer_start_dt", "load_dt", "n_load_versions", "in_neo4j_customer",
                     "first_edge_month", "last_edge_month"))
    p = table_location(T_CUST_DIM) if table_exists(T_CUST_DIM) else f"{LOC}/pkg_cust_dim"
    dim.write.mode("overwrite").option("compression", CODEC).parquet(p)
    spark.sql(f"DROP TABLE IF EXISTS {T_CUST_DIM}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in dim.schema.fields)
    spark.sql(f"CREATE EXTERNAL TABLE {T_CUST_DIM} ({cols}) STORED AS PARQUET LOCATION '{p}'")
    spark.catalog.refreshTable(T_CUST_DIM)

if BUILD_CUST_DIM or not table_exists(T_CUST_DIM):
    build_cust_dim()

# ---- required columns and key types (fail fast) ---------------------------------------------------------------------
def schema_of(key_or_table):
    if key_or_table in IN_LOC:
        ms = list_months(IN_LOC[key_or_table])
        if not ms:
            raise RuntimeError(f"no month folders under {IN_LOC[key_or_table]}")
        return spark.read.parquet(f"{IN_LOC[key_or_table]}/month={ms[-1]}").schema
    return spark.table(key_or_table).schema

NEED = {"edge_m": {"src_id": "string", "src_kind": None, "dst_id": "string", "dst_kind": None, "category": None,
                   "is_self": None, "amount": None, "n_txn": None},
        "edge_d": {"src_id": "string", "src_kind": None, "dst_id": "string", "dst_kind": None, "is_self": None, "txn_date": None},
        "cpty_month": {"cpty_id": "string", "fi": None, "amount": None},
        "cpm": {"ent_id": "string", "ent_name": None, "eligible": None, "hub_tier": None},
        T_CPTY_DIM: {"cpty_id": "string", "cpty_name": None, "fi": None, "amount": None},
        T_MAP: {"cpty_id": "string", "ent_l2": "string", "name_class": None},
        T_FI_ROUTING: {"routing": "string", "fi_canonical": None},
        T_FI_NAMEMAP: {"fi_raw": None, "fi_canonical": None},
        T_CUST_DIM: {"mdm_id": "string", "customer_name": None, "party_type": None, "naics2": None, "is_tm": None,
                     "customer_pwr_id": None, "master_pwr_id": None}}
problems = []
for src, cols in NEED.items():
    try:
        sch = {f.name.lower(): f.dataType.simpleString() for f in schema_of(src).fields}
    except Exception as e:
        problems.append(f"{src}: cannot read ({str(e).splitlines()[0][:120]})"); continue
    for c, typ in cols.items():
        if c not in sch:
            problems.append(f"{src}: column {c} missing")
        elif typ and sch[c] != typ:
            problems.append(f"{src}: {c} is {sch[c]}, expected {typ} (join keys must be strings)")
if problems:
    raise RuntimeError("input check failed:\n  " + "\n  ".join(problems))

STATIC_TABLES = [T_MAP, T_CPTY_DIM, T_FI_ROUTING, T_FI_NAMEMAP, T_CUST_DIM]
FP_STATIC = {t: path_fp(table_location(t)) for t in STATIC_TABLES}
CFG = {"v": METRICS_VERSION, "start": HISTORY_START, "lb": LOOKBACK, "rec": [REC_WINDOW, REC_MIN], "move": MOVE_SHARE,
       "naics": MIN_NAICS_CP, "hub": HUB_TIERS, "pnc": sorted(PNC_FI), "fi_tm": CUST_FI_TM_ONLY, "rails": RAILS,
       "rail_map": RAIL_OF_DIGIT, "legal": LEGAL}
SIG = md5({"cfg": CFG, "fp": FP_STATIC})
show(H(f"Inputs OK · signature {SIG}"),
     TB("Static inputs (fingerprint = data files : bytes : latest modification)",
        pd.DataFrame([{"table": t, "fingerprint": fp} for t, fp in FP_STATIC.items()])),
     TB("Monthly inputs", pd.DataFrame([{"input": k, "location": v, "months": len(list_months(v)),
                                          "first": (list_months(v) or ["—"])[0], "last": (list_months(v) or ["—"])[-1]}
                                         for k, v in IN_LOC.items()])))

In [ ]:
# §1b  Static lookups, written once per signature: counterparty id → entity, name class, name key, lifetime bank;
#      customer attributes; canonical bank tables
ST = {k: f"{STATIC}/{k}" for k in ("lk", "cust", "rout", "nm")}
_st_marker = read_json(f"{STATIC}/_PKG_STATIC.json")
if not (_st_marker and _st_marker.get("sig") == SIG and all(exists(f"{p}/_SUCCESS") for p in ST.values())):
    rm(STATIC)
    m1 = (spark.table(T_MAP).select("cpty_id", "ent_l2", "name_class")
               .groupBy("cpty_id").agg(F.min(F.struct("ent_l2", "name_class")).alias("_s"), F.count("*").alias("n_map_rows")))
    d1 = (spark.table(T_CPTY_DIM)
               .select("cpty_id", F.col("amount").cast("decimal(28,2)").alias("amount"), "cpty_name", "fi")
               .groupBy("cpty_id").agg(F.max(F.struct("amount", "cpty_name", "fi")).alias("_s"), F.count("*").alias("n_dim_rows")))
    lk = (m1.select("cpty_id", F.col("_s.ent_l2").alias("ent"), F.col("_s.name_class").alias("o_class"), "n_map_rows")
            .join(d1.select("cpty_id", F.col("_s.fi").alias("dim_fi"), name_key(F.col("_s.cpty_name")).alias("o_key"),
                            "n_dim_rows"), "cpty_id", "full"))
    lk.write.mode("overwrite").parquet(ST["lk"])
    cd = spark.table(T_CUST_DIM)
    (cd.select("mdm_id", F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"), "party_type", "naics2",
               name_key(F.col("customer_name")).alias("own_key"))
       .write.mode("overwrite").parquet(ST["cust"]))
    (spark.table(T_FI_ROUTING).where(F.col("fi_canonical").isNotNull())
          .select(F.col("routing").cast("string").alias("routing"), "fi_canonical").write.mode("overwrite").parquet(ST["rout"]))
    (spark.table(T_FI_NAMEMAP).where(F.col("fi_raw").isNotNull() & F.col("fi_canonical").isNotNull())
          .select("fi_raw", "fi_canonical").write.mode("overwrite").parquet(ST["nm"]))
    write_json(f"{STATIC}/_PKG_STATIC.json", {"sig": SIG, "built_at": dt.datetime.now().isoformat(timespec="seconds")})

LK, CUST, ROUT, NM = (spark.read.parquet(ST[k]) for k in ("lk", "cust", "rout", "nm"))
q = LK.agg(F.count("*").alias("ids"), F.sum((F.col("n_map_rows") > 1).cast("int")).alias("map_dup_ids"),
           F.sum((F.col("n_dim_rows") > 1).cast("int")).alias("dim_dup_ids"),
           F.sum(F.col("ent").isNull().cast("int")).alias("in_dim_not_map"),
           F.sum(F.col("n_dim_rows").isNull().cast("int")).alias("in_map_not_dim"),
           F.countDistinct("ent").alias("entities")).toPandas()
uniq = []
for name, df, key in (("customer dim", CUST, "mdm_id"), ("routing", ROUT, "routing"), ("FI name map", NM, "fi_raw")):
    r = df.agg(F.count("*").alias("rows"), F.countDistinct(key).alias("keys")).first()
    uniq.append({"lookup": name, "rows": r["rows"], "distinct keys": r["keys"]})
uniq = pd.DataFrame(uniq)
bad = uniq[uniq["rows"] != uniq["distinct keys"]]
if len(bad):
    raise RuntimeError(f"lookups must be one row per key: {bad.to_dict('records')}")
BCAST_ROUT, BCAST_NM = (int(uniq.loc[uniq.lookup == n, "rows"].iloc[0]) <= 2_000_000 for n in ("routing", "FI name map"))

def canon(df, id_col, raw_col, out="fi_c"):
    """Canonical bank: routing prefix of the id → pkg_fi_routing, else raw FI name → pkg_fi_name_map, else the raw name."""
    rt = ROUT.select(F.col("routing").alias("_rt"), F.col("fi_canonical").alias("_fc_rt"))
    nm = NM.select(F.col("fi_raw").alias("_fr"), F.col("fi_canonical").alias("_fc_nm"))
    rt, nm = (F.broadcast(rt) if BCAST_ROUT else rt), (F.broadcast(nm) if BCAST_NM else nm)
    return (df.withColumn("_rt", F.regexp_extract(F.col(id_col), r"^(\d{9})-", 1))
              .join(rt, "_rt", "left")
              .join(nm, F.col(raw_col) == F.col("_fr"), "left")
              .withColumn(out, F.coalesce("_fc_rt", "_fc_nm", raw_col))
              .drop("_rt", "_fc_rt", "_fr", "_fc_nm"))

# PNC as holding bank: the canonical names must exist, or pnc_fi_share is silently 0
pnc_like = (ROUT.select(F.col("fi_canonical").alias("fi_c"), F.lit("routing").alias("src"))
                .unionByName(NM.select(F.col("fi_canonical").alias("fi_c"), F.lit("name map").alias("src")))
                .where(F.upper("fi_c").contains("PNC")).groupBy("fi_c")
                .agg(F.sum((F.col("src") == "routing").cast("int")).alias("routings"),
                     F.sum((F.col("src") == "name map").cast("int")).alias("raw_names"))
                .orderBy(F.desc("routings")).limit(10).toPandas())
missing_pnc = [n for n in PNC_FI if n not in set(pnc_like.fi_c)]
show(H("Static lookups"),
     TB("Counterparty ids (map v2 × pkg_cpty_dim)", q),
     N("map_dup_ids: ids with several map rows, reduced to the lowest ent_l2. dim_dup_ids: ids with several dim rows, "
       "reduced to the largest-amount row. in_dim_not_map: ids that would fall back to their raw id."),
     TB("One row per key", uniq),
     TB("Canonical bank names containing PNC (routings and raw names mapped to each)", pnc_like),
     BAD(f"PNC_FI names not found among canonical banks: {missing_pnc}. Set PNC_FI to the names above (or [] if PNC "
         f"never appears as a holding bank) and rerun from here.") if missing_pnc else OK(f"PNC_FI found: {PNC_FI}"))
if missing_pnc:
    raise RuntimeError(f"PNC_FI {missing_pnc} not among the canonical bank names (see the table above)")

## §1b Pre-flight: the 2026-07 / 2026-08 dollar jump

Monthly edge totals jumped to about 3.4T and 2.9T in 2026-07 and 2026-08, from 1.6–1.9T before. The metrics stay
faithful to the edges either way. This decides only whether those months need a caveat (or a filter later) in dollar
rankings and spike rules. If the jump is a handful of new pairs on one category, it is most likely a new feed. If the
same customers grew across many counterparties, it is real activity.

In [ ]:
# §1c  Pre-flight: totals by rail and category, customers driving the latest months, largest single flows
if PREFLIGHT:
    AV_EM = list_months(IN_LOC["edge_m"])
    PM = AV_EM[-PREFLIGHT_MONTHS:]
    e = (spark.read.parquet(IN_LOC["edge_m"]).where(F.col("month").isin(PM))
              .withColumn("rail", F.element_at(F.create_map(*[F.lit(x) for kv in RAIL_OF_DIGIT.items() for x in kv]),
                                               F.substring("category", 1, 1))))
    B = 1e9
    by_cat = e.groupBy("category", "rail", "month").agg(F.sum("amount").cast("double").alias("a")).toPandas()   # one scan
    by_rail = by_cat.pivot_table(index="month", columns="rail", values="a", aggfunc="sum").fillna(0) / B
    by_rail.insert(0, "total_usd_bn", by_rail.sum(axis=1))
    cat = by_cat.pivot_table(index="category", columns="month", values="a", aggfunc="sum").fillna(0) / B
    base_cols = PM[:-2] if len(PM) > 2 else PM[:1]
    cat["prior_median"] = cat[base_cols].median(axis=1)
    cat["jump_last2_vs_prior"] = cat[PM[-2:]].mean(axis=1) - cat["prior_median"]
    cat = cat.sort_values("jump_last2_vs_prior", ascending=False).head(10).reset_index()
    # customers: largest month-over-prior jump (both directions)
    cm = (e.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("mdm_id"), "month", "amount")
           .unionByName(e.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("mdm_id"), "month", "amount"))
           .groupBy("mdm_id", "month").agg(F.sum("amount").cast("double").alias("a")))
    last2 = cm.where(F.col("month").isin(PM[-2:])).groupBy("mdm_id").agg(F.avg("a").alias("last2_avg"))
    prior = (cm.where(F.col("month").isin(base_cols)).groupBy("mdm_id")
               .agg(F.expr("percentile_approx(a, 0.5)").alias("prior_median"), F.count("*").alias("prior_months")))
    top_c = (last2.join(prior, "mdm_id", "left").withColumn("jump", F.col("last2_avg") - F.coalesce("prior_median", F.lit(0.0)))
                  .orderBy(F.desc("jump")).limit(12)
                  .join(spark.table(T_CUST_DIM).select("mdm_id", "customer_name", "naics_desc", "is_tm"), "mdm_id", "left")
                  .toPandas())
    for c in ("last2_avg", "prior_median", "jump"):
        top_c[c] = top_c[c] / B
    top_c = top_c.rename(columns={"last2_avg": "last2_avg_usd_bn", "prior_median": "prior_median_usd_bn", "jump": "jump_usd_bn"})
    # largest single monthly edges in the last two months (collected once, so every join sees the same 15 rows),
    # and whether the pair existed earlier in the window
    bigp = (e.where(F.col("month").isin(PM[-2:])).orderBy(F.desc("amount"), "src_id", "dst_id", "category").limit(15)
             .select("month", "src_id", "src_kind", "dst_id", "dst_kind", "category", "rail", "amount", "n_txn").toPandas())
    small = F.broadcast(spark.createDataFrame(bigp[["src_id", "dst_id"]].drop_duplicates()))
    seen_p = (e.join(small, ["src_id", "dst_id"], "inner").groupBy("src_id", "dst_id")
               .agg(F.min("month").alias("first_month_in_window"), F.countDistinct("month").alias("months_in_window")).toPandas())
    ids = F.broadcast(spark.createDataFrame(pd.DataFrame({"_id": pd.unique(bigp[["src_id", "dst_id"]].values.ravel())})))
    names = dict((spark.table(T_CPTY_DIM).select(F.col("cpty_id").alias("_id"), F.col("cpty_name").alias("_nm"))
                  .unionByName(spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("_id"), F.col("customer_name").alias("_nm")))
                  .join(ids, "_id").groupBy("_id").agg(F.min("_nm").alias("_nm")).toPandas().values))
    bigp = bigp.merge(seen_p, on=["src_id", "dst_id"], how="left")
    bigp["payer"], bigp["payee"] = bigp.src_id.map(names), bigp.dst_id.map(names)
    bigp["amount_usd_bn"] = bigp["amount"].astype(float) / B
    bigp = bigp[["month", "payer", "src_kind", "payee", "dst_kind", "category", "rail", "amount_usd_bn", "n_txn",
                 "first_month_in_window", "months_in_window"]]
    show(H(f"Pre-flight · {PM[0]} → {PM[-1]}"),
         TB("Edge dollars by rail (USD bn)", by_rail.reset_index()),
         TB("Categories with the largest rise: mean of the last two months minus the prior median (USD bn)", cat),
         TB("Customers with the largest rise (both directions, USD bn)", top_c),
         TB("Largest single monthly edges in the last two months", bigp),
         N("New feed: the rise sits in one category, on pairs first seen in the last months, for a few customers. "
           "Real activity: established pairs and customers that grew. Either way the build below is unaffected."))

## §2 Column dictionary (single source of truth)

Column order, type, family, scope, direction, definition and empty-value rule for the three output tables. The final
select of every month casts to these types, so every month has the same schema. `since` says which version added the
column; `v3_change` says what changed for columns that already existed in v2.

In [ ]:
# §2  Dictionary
D, S3, S2 = ["in", "out"], ["int", "ext", "self"], ["int", "ext"]
DEC, DBL, LNG, STR, INT, BOOL = "decimal(22,2)", "double", "bigint", "string", "int", "boolean"
SCOPE_TXT = {"int": "other PNC customers", "ext": "external counterparties", "self": "the customer's own accounts (same mdm_id)",
             "all": "PNC customers + external counterparties (self-transfers excluded)"}
DIR_TXT, WITH_TXT = {"in": "received", "out": "paid"}, {"in": "from", "out": "to"}
E_ZERO, E_SHARE, E_NAME = "0 when no activity", "NULL when the denominator is 0", "NULL when there is nothing to name"
ENT = "counterparty entity = ent_l2 of pkg_cpty_entity_map_v2 (= ent_id of pkg_cpty_metrics_monthly)"
CANON = "canonical bank (pkg_fi_routing via the id's routing prefix, else pkg_fi_name_map, else the raw name)"
DICT = []
def col(c, fam, scope, d, typ, definition, empty, since="v2", change="", table="metrics"):
    DICT.append({"table": table, "column": c, "family": fam, "scope": scope, "direction": d, "type": typ,
                 "definition": definition, "empty_value": empty, "since": since, "v3_change": change})

# ---- metrics table -------------------------------------------------------------------------------------------------
for s in ["int", "ext", "self", "all"]:
    for d in D:
        col(f"amt_{s}_{d}", "flow", s, d, DEC, f"dollars {DIR_TXT[d]}, {SCOPE_TXT[s]}", E_ZERO)
        col(f"ntxn_{s}_{d}", "flow", s, d, LNG, f"transactions {DIR_TXT[d]}, {SCOPE_TXT[s]}", E_ZERO)
for d in D:
    col(f"avg_ticket_{d}", "flow", "all", d, DBL, f"amt_all_{d} / ntxn_all_{d}: average payment size {DIR_TXT[d]}", E_SHARE)
    col(f"active_days_{d}", "flow", "all", d, LNG, f"distinct posting dates with at least one transaction {DIR_TXT[d]} (self excluded)", E_ZERO)
    col(f"ext_share_{d}", "flow", "all", d, DBL, f"share of dollars {DIR_TXT[d]} that involve external counterparties", E_SHARE)
col("net_flow_all", "flow", "all", "", DEC, "amt_all_in − amt_all_out", E_ZERO)
col("net_flow_ext", "flow", "ext", "", DEC, "amt_ext_in − amt_ext_out", E_ZERO)
col("passthrough_ratio", "flow", "all", "", DBL, "min(amt_all_in, amt_all_out) / max(...): 1 = everything received is paid out", E_SHARE)
for s in ["int", "ext", "all"]:
    for d in D:
        col(f"ncpty_{s}_{d}", "breadth", s, d, LNG, f"distinct counterparties {DIR_TXT[d]} {WITH_TXT[d]}, {SCOPE_TXT[s]}"
            + (f"; external at {ENT}" if s != "int" else ""), E_ZERO, change="external now ent_l2 (v2: ent_l1)" if s != "int" else "")
for d in D:
    col(f"top1_share_{d}", "concentration", "all", d, DBL, f"share of dollars {DIR_TXT[d]} with the single largest counterparty", E_SHARE)
    col(f"hhi_{d}", "concentration", "all", d, DBL, f"Herfindahl index of dollars {DIR_TXT[d]} across counterparties (1 = one counterparty)", E_SHARE)
for s in S2:
    for d in D:
        x = f"({SCOPE_TXT[s]})"
        ch = "ent_l2" if s == "ext" else ""
        col(f"new_cpty_{s}_{d}", "churn", s, d, LNG, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} this month, not seen in the previous {LOOKBACK} months {x}", E_ZERO, change=ch)
        col(f"reactivated_cpty_{s}_{d}", "churn", s, d, LNG, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} this month, seen in the previous {LOOKBACK} months but not last month {x}", E_ZERO, change=ch)
        col(f"retained_cpty_{s}_{d}", "churn", s, d, LNG, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} both last month and this month {x}", E_ZERO, change=ch)
        col(f"lost_cpty_{s}_{d}", "churn", s, d, LNG, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} last month but not this month {x}", E_ZERO, change=ch)
for d in D:
    col(f"retention_{d}", "churn", "all", d, DBL, f"retained / last month's counterparties, {DIR_TXT[d]}", E_SHARE)
    col(f"new_amt_share_{d}", "churn", "all", d, DBL, f"share of this month's dollars {DIR_TXT[d]} with new counterparties", E_SHARE)
    col(f"lost_amt_share_{d}", "churn", "all", d, DBL, f"share of last month's dollars {DIR_TXT[d]} that were with counterparties lost this month", E_SHARE)
col("history_months", "churn", "", "", INT, f"months of history behind the churn columns (below {LOOKBACK} during the first {LOOKBACK} months after {HISTORY_START})", "never empty")
for d in D:
    col(f"rec_standing_{d}", "recurring", "all", d, LNG, f"standing relationships {DIR_TXT[d]}: present in ≥ {REC_MIN} of the previous {REC_WINDOW} months", E_ZERO)
    col(f"rec_broken_{d}", "recurring", "all", d, LNG, f"standing relationships {DIR_TXT[d]} absent this month", E_ZERO)
    col(f"rec_broken_share_{d}", "recurring", "all", d, DBL, f"rec_broken / rec_standing, {DIR_TXT[d]}", E_SHARE)
    col(f"rec_broken_amt_share_{d}", "recurring", "all", d, DBL, f"share of standing dollars {DIR_TXT[d]} (last monthly amount) whose relationship broke", E_SHARE)
for d in D:
    b = f"{DIR_TXT[d]} {WITH_TXT[d]}"
    col(f"n_fi_{d}", "banks", "ext", d, LNG, f"distinct {CANON}s of external counterparties {b}, all rails", E_ZERO,
        change="canonical name, the id's bank this month, counted at id level (v2: lifetime bank of the pair's first id)")
    col(f"n_fi_ach_{d}", "banks", "ext", d, LNG, f"distinct canonical banks {b} on ACH only (attrition's fin_{d}_n definition)", E_ZERO, change="canonical name, monthly bank")
    col(f"top_fi_{d}", "banks", "ext", d, STR, f"canonical bank with the most external dollars {DIR_TXT[d]} (ties: alphabetical last)", E_NAME, since="v3")
    col(f"top_fi_share_{d}", "banks", "ext", d, DBL, f"share of bank-identified external dollars {DIR_TXT[d]} through the largest bank", E_SHARE, change="canonical name")
    col(f"fi_coverage_{d}", "banks", "ext", d, DBL, f"share of external dollars {DIR_TXT[d]} whose counterparty bank is known", E_SHARE)
    col(f"pnc_fi_share_{d}", "banks", "ext", d, DBL, f"share of bank-identified external dollars {DIR_TXT[d]} whose counterparty account is at PNC (PNC_FI: converted accounts not mapped to a customer); 1 − this = competitor share", E_SHARE, since="v3")
for d in D:
    b = f"{DIR_TXT[d]} {WITH_TXT[d]}"
    col(f"new_fi_{d}", "bank churn", "ext", d, LNG, f"banks {b} this month, not seen in the previous {LOOKBACK} months", E_ZERO, change="canonical name")
    col(f"reactivated_fi_{d}", "bank churn", "ext", d, LNG, f"banks seen in the previous {LOOKBACK} months but not last month, back this month ({DIR_TXT[d]})", E_ZERO, change="canonical name")
    col(f"retained_fi_{d}", "bank churn", "ext", d, LNG, f"banks {b} both last month and this month", E_ZERO, change="canonical name")
    col(f"lost_fi_{d}", "bank churn", "ext", d, LNG, f"banks {b} last month but not this month", E_ZERO, change="canonical name")
    col(f"fi_retention_{d}", "bank churn", "ext", d, DBL, f"retained_fi / last month's banks, {DIR_TXT[d]}", E_SHARE)
    col(f"fi_rec_broken_{d}", "bank churn", "ext", d, LNG, f"standing bank relationships ({REC_MIN} of {REC_WINDOW} months) absent this month, {DIR_TXT[d]}", E_ZERO, change="canonical name")
    col(f"fi_rec_broken_share_{d}", "bank churn", "ext", d, DBL, f"fi_rec_broken / standing bank relationships, {DIR_TXT[d]}", E_SHARE)
for d in D:
    col(f"cpty_bank_moved_{d}", "bank moves", "ext", d, LNG, f"counterparties {DIR_TXT[d]} {WITH_TXT[d]} last month and this month whose main bank changed (the counterparty moved or re-routed its account)", E_ZERO, since="v3")
    col(f"lost_fi_cpty_moved_{d}", "bank moves", "ext", d, LNG, f"lost banks (lost_fi_{d}) where ≥ {MOVE_SHARE:.0%} of last month's dollars at that bank now flow with the same counterparties at other banks: driven by the counterparty, not the customer", E_ZERO, since="v3")
    col(f"fi_rec_broken_cpty_moved_{d}", "bank moves", "ext", d, LNG, f"broken standing banks (fi_rec_broken_{d}) whose disappearance is explained by the counterparties moving: ≥ {MOVE_SHARE:.0%} of the last dollars of the counterparties last seen at that bank flow with them at other banks, this month or any month since the bank was last active", E_ZERO, since="v3")
    col(f"fi_rec_broken_net_{d}", "bank moves", "ext", d, LNG, f"fi_rec_broken_{d} − fi_rec_broken_cpty_moved_{d}: standing-bank breaks not explained by counterparties moving (the attrition-facing count)", E_ZERO, since="v3")
    col(f"fi_rec_broken_net_share_{d}", "bank moves", "ext", d, DBL, f"fi_rec_broken_net_{d} / standing bank relationships", E_SHARE, since="v3")
for d in D:
    col(f"selfpay_amt_{d}", "self-pay", "ext", d, DEC, f"dollars {DIR_TXT[d]} {WITH_TXT[d]} counterparties with the customer's own name at other banks (exact normalized name, or first 8 characters for organisations)", E_ZERO)
    col(f"selfpay_share_{d}", "self-pay", "ext", d, DBL, f"selfpay_amt_{d} / amt_ext_{d}", E_SHARE)
    col(f"selfpay_n_fi_{d}", "self-pay", "ext", d, LNG, f"distinct canonical banks holding own-name accounts, {DIR_TXT[d]}", E_ZERO, change="canonical name")
    col(f"selfpay_top_fi_{d}", "self-pay", "ext", d, STR, f"canonical bank holding the most own-name dollars, {DIR_TXT[d]}", E_NAME, since="v3")
for d in D:
    col(f"ext_business_share_{d}", "composition", "ext", d, DBL, f"share of external dollars {DIR_TXT[d]} with counterparties typed business by typer v2", E_SHARE, change="typer v2 name_class (v2: is_business of pkg_cpty_dim)")
    col(f"ext_person_share_{d}", "composition", "ext", d, DBL, f"share of external dollars {DIR_TXT[d]} with counterparties typed person by typer v2", E_SHARE, since="v3")
    col(f"int_tm_share_{d}", "composition", "int", d, DBL, f"share of internal dollars {DIR_TXT[d]} with TM customers", E_SHARE)
    col(f"int_org_share_{d}", "composition", "int", d, DBL, f"share of internal dollars {DIR_TXT[d]} with organisations (party_type O)", E_SHARE)
for d in D:
    col(f"n_naics2_{d}", "sector", "int", d, LNG, f"distinct NAICS2 sectors of internal counterparties (NULL below {MIN_NAICS_CP} counterparties with a valid NAICS)", f"NULL below {MIN_NAICS_CP} counterparties")
    col(f"naics2_top_share_{d}", "sector", "int", d, DBL, "dollar share of the largest counterparty sector (same guard)", f"NULL below {MIN_NAICS_CP} counterparties")
    col(f"naics2_entropy_{d}", "sector", "int", d, DBL, "Shannon entropy (nats) of dollars across counterparty sectors (same guard)", f"NULL below {MIN_NAICS_CP} counterparties")
    col(f"same_naics2_share_{d}", "sector", "int", d, DBL, "share of sector-known internal dollars with counterparties in the customer's own NAICS2", E_SHARE + " or own NAICS unknown")
for d in D:
    for r in RAILS:
        col(f"share_{r}_{d}", "rails", "all", d, DBL, f"share of dollars {DIR_TXT[d]} on {r}", E_SHARE)
    col(f"n_rails_{d}", "rails", "all", d, LNG, f"number of rails used, {DIR_TXT[d]}", E_ZERO)
for d in D:
    for k in HUB_TIERS:
        col(f"n_hub_t{k}_{d}", "hub exposure", "ext", d, LNG, f"external counterparties {DIR_TXT[d]} {WITH_TXT[d]} with hub_tier ≥ {k} in pkg_cpty_metrics_monthly this month", E_ZERO, since="v3")
    for k in HUB_TIERS:
        col(f"hub_t{k}_share_{d}", "hub exposure", "ext", d, DBL, f"share of external dollars {DIR_TXT[d]} with counterparties of hub_tier ≥ {k}", E_SHARE, since="v3")
for d in D:
    col(f"n_cpty_elig_{d}", "cpty links", "ext", d, LNG, f"external counterparties {DIR_TXT[d]} {WITH_TXT[d]} that are eligible (full metric set) in pkg_cpty_metrics_monthly this month", E_ZERO, since="v3")
    col(f"elig_share_{d}", "cpty links", "ext", d, DBL, f"share of external dollars {DIR_TXT[d]} with eligible counterparties", E_SHARE, since="v3")

IDENTITY_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
                 "naics2", "naics_status", "is_tm", "pwr_shared", "family_size", "customer_start_dt", "in_neo4j_customer"]
IDENTITY_DEF = {
    "mdm_id": "MDM party id (bank-wide customer key)",
    "customer_pwr_id": "TM customer power id; NULL = not a TM customer",
    "master_pwr_id": "ultimate parent power id of the customer's family",
    "customer_name": "customer name (latest load_dt)",
    "party_type": "P = person, O = organisation",
    "naics_cd": "NAICS code as recorded (populated for TM customers)",
    "naics_desc": "NAICS description",
    "naics2": "2-digit NAICS sector, only when naics_status = valid",
    "naics_status": "valid / placeholder (******, UNKNOWN, …) / missing",
    "is_tm": "true when the customer has a customer_pwr_id",
    "pwr_shared": "true when this customer_pwr_id is also held by another mdm_id (joins on power id are ambiguous)",
    "family_size": "number of customers sharing this master_pwr_id",
    "customer_start_dt": "customer start date",
    "in_neo4j_customer": "false when the mdm_id appears in payments but not in neo4j_customer (identity columns then NULL)"}
_cd_types = {f.name: f.dataType.simpleString() for f in spark.table(T_CUST_DIM).schema.fields}
METRIC_COLS = [r["column"] for r in DICT if r["table"] == "metrics"]
METRIC_TYPES = {r["column"]: r["type"] for r in DICT if r["table"] == "metrics"}
ident_rows = [{"table": "metrics", "column": c, "family": "identity", "scope": "", "direction": "", "type": _cd_types.get(c, STR),
               "definition": IDENTITY_DEF[c], "empty_value": "NULL when not in neo4j_customer" if c not in ("mdm_id", "is_tm", "pwr_shared", "in_neo4j_customer") else "never empty",
               "since": "v2", "v3_change": ""} for c in IDENTITY_COLS]
DICT = ident_rows + DICT
col("month", "partition", "", "", STR, "activity month YYYY-MM", "never empty")

# ---- pair table ----------------------------------------------------------------------------------------------------
PAIR_SPEC = [("cust", STR, "customer mdm_id"), ("dir", STR, "in = received, out = paid"),
             ("scope", STR, "int / ext / self"), ("other", STR, f"counterparty: mdm_id for int/self, {ENT} for ext"),
             ("amount", DEC, "dollars"), ("n_txn", LNG, "transactions"),
             ("o_fi", STR, f"main {CANON} of the pair this month (largest dollars; ext only)"),
             ("n_fi", LNG, "distinct canonical banks of the pair this month (ext only)"),
             ("o_class", STR, "typer v2 name class of the pair's largest id (ext only)"),
             ("o_biz", BOOL, "o_class = business (ext only)"), ("o_self", BOOL, "own-name counterparty (self-pay match, ext only)"),
             ("o_is_tm", BOOL, "counterparty is a TM customer (int only)"), ("o_party_type", STR, "counterparty party type (int only)"),
             ("o_naics2", STR, "counterparty NAICS2 (int only)")]
for c_, t_, d_ in PAIR_SPEC:
    col(c_, "pair", "", "", t_, d_, "NULL when not applicable", since="v3" if c_ in ("n_fi", "o_class") else "v2",
        change={"other": "ext now ent_l2", "o_fi": "canonical, largest-dollar bank (v2: first bank, not deterministic)"}.get(c_, ""),
        table="pairs")
col("month", "partition", "", "", STR, "activity month YYYY-MM", "never empty", since="v2", table="pairs")

# ---- customer × bank table -----------------------------------------------------------------------------------------
FI_SPEC = [("mdm_id", STR, "customer mdm_id"), ("customer_pwr_id", STR, "TM customer power id"),
           ("master_pwr_id", STR, "ultimate parent power id"), ("customer_name", STR, "customer name"),
           ("dir", STR, "in = received from counterparties at this bank, out = paid to them"),
           ("fi", STR, CANON),
           ("status", STR, "new / reactivated / retained (active this month), lost (active last month only), lapsed (standing, absent last month and this month)"),
           ("active", BOOL, "any transaction with a counterparty at this bank this month"),
           ("in_prev", BOOL, "active last month"), ("standing", BOOL, f"active in ≥ {REC_MIN} of the previous {REC_WINDOW} months"),
           ("rec_broken", BOOL, "standing and not active this month"),
           ("cpty_moved", BOOL, f"not active, and its disappearance is explained by the counterparties moving: ≥ {MOVE_SHARE:.0%} of their last dollars at this bank flow at other banks (this month or since the bank was last active)"),
           ("is_pnc", BOOL, "bank is in PNC_FI"),
           ("amount", DEC, "dollars this month (0 when not active)"), ("n_txn", LNG, "transactions this month"),
           ("n_cpty", LNG, "counterparty entities at this bank this month"), ("ntxn_ach", LNG, "ACH transactions this month"),
           ("selfpay_amt", DEC, "dollars with own-name counterparties at this bank this month"),
           ("last_amt", DBL, "dollars in the last month the bank was active (for lost / lapsed rows)")]
for c_, t_, d_ in FI_SPEC:
    col(c_, "customer × bank", "ext", "", t_, d_, "0 / false when not active", since="v3", table="cust_fi")
col("month", "partition", "", "", STR, "activity month YYYY-MM", "never empty", since="v3", table="cust_fi")
DICT_DF = pd.DataFrame(DICT)
DICT_DF["table"] = DICT_DF["table"].map({"metrics": OUT["metrics"], "pairs": OUT["pairs"], "cust_fi": OUT["cust_fi"]})
assert not DICT_DF.duplicated(["table", "column"]).any(), DICT_DF[DICT_DF.duplicated(["table", "column"], keep=False)]
_dd = spark.createDataFrame(DICT_DF.astype(str))
_dd.write.mode("overwrite").option("compression", CODEC).parquet(OUT_P["dict"])
spark.sql(f"DROP TABLE IF EXISTS {OUT['dict']}")
spark.sql(f"CREATE EXTERNAL TABLE {OUT['dict']} (" + ", ".join(f"`{c}` string" for c in DICT_DF.columns)
          + f") STORED AS PARQUET LOCATION '{OUT_P['dict']}'")
spark.catalog.refreshTable(OUT["dict"])
summ = (DICT_DF.groupby(["table", "since"]).column.count().unstack(fill_value=0).reset_index())
new_cols = DICT_DF[(DICT_DF.since == "v3") & (DICT_DF.table == OUT["metrics"])].groupby("family").column.apply(
    lambda s: ", ".join(sorted({x.rsplit("_", 1)[0] if x.endswith(("_in", "_out")) else x for x in s}))).reset_index()
show(H(f"Dictionary written to {OUT['dict']}"), TB("Columns by table and version", summ),
     TB("New metric families in v3 (per direction _in / _out)", new_cols))

## §3 Monthly build

Each month runs these steps; each step's output is saved under the work folder and reused if the month is restarted:

1. **`lk`:** every counterparty id active this month gets its entity (`ent_l2`), typer v2 class, name key and
   canonical bank (the id's bank this month from `pkg_cpty_month`, else its lifetime bank).
2. **`l1`:** each edge read from each customer side, aggregated to customer × direction × scope × entity × bank,
   with dollars by rail and by counterparty type. Every later step reads this, not the edges.
3. **`pairs`:** customer × direction × scope × entity, with the pair's main bank. Staged as the pair table.
4. **`cpty`:** churn state for relationships (12-bit presence mask, last amount, last main bank) → new / reactivated /
   retained / lost, standing / broken, counterparties whose bank changed, and per lost bank the dollars that moved.
5. **`fi`:** churn state for banks → bank churn, standing / broken banks, counterparty-driven flags. Staged as
   `pkg_cust_fi_month`.
6. **`hub`:** external pairs joined to `pkg_cpty_metrics_monthly` for the month → hub exposure and eligible share.
7. **`days`, `sector`:** active days from the daily edges, sector mix of internal counterparties.
8. **Assemble** → stage → **reconcile exactly** (dollars and transactions per direction against the edges, one row per
   customer, pair and bank totals) → move into place → write the month marker.

A month is done when its marker carries the current signature, the fingerprint of its four monthly inputs, and the
build id of the previous month. Rebuilding a month therefore invalidates every later month.

In [ ]:
# §3a  Step functions
RAIL_MAP = F.create_map(*[F.lit(x) for kv in RAIL_OF_DIGIT.items() for x in kv])
FULL_MASK, REC_MASK = (1 << LOOKBACK) - 1, (1 << REC_WINDOW) - 1
CPTY_STATE_SCHEMA = "cust string, dir string, scope string, other string, mask int, last_amt double, last_fi string"
FI_STATE_SCHEMA = "cust string, dir string, fi string, mask int, last_amt double, moved boolean"
month_in = lambda key, m: f"{IN_LOC[key]}/month={m}"

def month_fp(m):
    return md5({k: path_fp(month_in(k, m)) for k in ("edge_m", "edge_d", "cpty_month", "cpm")})

def edges(m):
    return spark.read.parquet(month_in("edge_m", m))

def s_lk(m):
    em = edges(m)
    ids = (em.where(F.col("src_kind") != "CUST").select(F.col("src_id").alias("cpty_id"))
             .unionByName(em.where(F.col("dst_kind") != "CUST").select(F.col("dst_id").alias("cpty_id"))).distinct())
    mfi = (spark.read.parquet(month_in("cpty_month", m)).where(F.col("fi").isNotNull())
                .groupBy("cpty_id", "fi").agg(F.sum("amount").alias("a"))
                .groupBy("cpty_id").agg(F.max(F.struct("a", "fi")).alias("_s"))        # deterministic: ties → name order
                .select("cpty_id", F.col("_s.fi").alias("m_fi")))
    x = (ids.join(LK.drop("n_map_rows", "n_dim_rows"), "cpty_id", "left").join(mfi, "cpty_id", "left")
            .withColumn("map_miss", F.col("ent").isNull())
            .withColumn("other", F.coalesce("ent", "cpty_id"))
            .withColumn("fi_raw", F.coalesce("m_fi", "dim_fi")))
    return canon(x, "cpty_id", "fi_raw").select("cpty_id", "other", "o_class", "o_key", "fi_c", "map_miss")

def s_l1(m, lk):
    em = edges(m)
    keep = ["category", "is_self", "amount", "n_txn"]
    out_ = em.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"),
                                                       F.col("dst_id").alias("oid"), F.col("dst_kind").alias("okind"), *keep)
    in_ = em.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"),
                                                      F.col("src_id").alias("oid"), F.col("src_kind").alias("okind"), *keep)
    p = (out_.unionByName(in_)
             .withColumn("scope", F.when(F.col("is_self"), "self").when(F.col("okind") == "CUST", "int").otherwise("ext"))
             .withColumn("rail", F.element_at(RAIL_MAP, F.substring("category", 1, 1))))
    own = CUST.select(F.col("mdm_id").alias("cust"), "own_key", F.col("party_type").alias("own_party"))
    ext = (p.where(F.col("scope") == "ext")
            .join(lk.withColumnRenamed("cpty_id", "oid"), "oid", "left")
            .join(own, "cust", "left")
            .withColumn("o_self", F.coalesce(
                (F.col("own_key") == F.col("o_key")) |
                ((F.col("own_party") == "O") & (F.length("own_key") >= 8) & (F.length("o_key") >= 8)
                 & (F.substring("own_key", 1, 8) == F.substring("o_key", 1, 8))), F.lit(False)))
            .drop("own_key", "own_party", "o_key", "map_miss"))
    intr = (p.where(F.col("scope") != "ext").withColumn("other", F.col("oid"))
             .join(CUST.select(F.col("mdm_id").alias("other"), F.col("is_tm").alias("o_is_tm"),
                               F.col("party_type").alias("o_party_type"), F.col("naics2").alias("o_naics2")), "other", "left"))
    rows = ext.unionByName(intr, allowMissingColumns=True)
    a = F.col("amount").cast("double")
    sw = lambda cond: F.sum(F.when(cond, a).otherwise(0.0))
    return (rows.groupBy("cust", "dir", "scope", "other", "fi_c").agg(
                F.sum("amount").cast(DEC).alias("amount"), F.sum("n_txn").cast("long").alias("n_txn"),
                *[sw(F.col("rail") == r).alias(f"amt_{r}") for r in RAILS],
                F.sum(F.when(F.col("rail") == "ACH", F.col("n_txn")).otherwise(0)).cast("long").alias("ntxn_ach"),
                sw(F.col("o_class") == "business").alias("amt_biz"), sw(F.col("o_class") == "person").alias("amt_person"),
                sw(F.col("o_self")).alias("amt_selfpay"), sw(F.col("o_is_tm")).alias("amt_tm"),
                sw(F.col("o_party_type") == "O").alias("amt_org"),
                F.max(F.struct(F.col("amount"), F.col("o_class"))).alias("_cl"),
                F.max(F.col("o_self").cast("int")).alias("o_self"), F.max(F.col("o_is_tm").cast("int")).alias("o_is_tm"),
                F.max("o_party_type").alias("o_party_type"), F.max("o_naics2").alias("o_naics2"))
            .withColumn("o_class", F.col("_cl.o_class")).drop("_cl"))

def s_pairs(l1):
    g = l1.groupBy("cust", "dir", "scope", "other").agg(
        F.sum("amount").cast(DEC).alias("amount"), F.sum("n_txn").cast("long").alias("n_txn"),
        F.max(F.when(F.col("fi_c").isNotNull(), F.struct("amount", "fi_c"))).alias("_fi"),
        F.count("fi_c").alias("n_fi"), F.max(F.struct("amount", "o_class")).alias("_cl"),
        F.max("o_self").alias("o_self"), F.max("o_is_tm").alias("o_is_tm"),
        F.max("o_party_type").alias("o_party_type"), F.max("o_naics2").alias("o_naics2"))
    g = g.select("cust", "dir", "scope", "other", "amount", "n_txn", F.col("_fi.fi_c").alias("o_fi"), "n_fi",
                 F.col("_cl.o_class").alias("o_class"), (F.col("_cl.o_class") == "business").alias("o_biz"),
                 "o_self", "o_is_tm", "o_party_type", "o_naics2")
    return g.select(*[F.col(c_).cast(t_).alias(c_) for c_, t_, _ in PAIR_SPEC])

def load_state(kind, m, prev_build):
    """State as of the end of month m-1; empty at HISTORY_START. Must belong to the published build of m-1."""
    if m == HISTORY_START:
        return spark.createDataFrame([], CPTY_STATE_SCHEMA if kind == "cpty" else FI_STATE_SCHEMA)
    pm, mk = prev_month(m), read_json(f"{state_path(kind, prev_month(m))}/_PKG_STATE.json")
    if not mk or mk.get("build_id") != prev_build or not exists(f"{state_path(kind, pm)}/_SUCCESS"):
        raise RuntimeError(f"churn state ({kind}) for {pm} is missing or belongs to another build. Months are built in "
                           f"order and only {KEEP_STATES} states are kept: set REBUILD_FROM = HISTORY_START "
                           f"(or the earliest month whose previous state exists) and rerun")
    return spark.read.parquet(state_path(kind, pm))

def write_state(df, kind, m, build_id):
    df.write.mode("overwrite").parquet(state_path(kind, m))
    write_json(f"{state_path(kind, m)}/_PKG_STATE.json", {"build_id": build_id, "sig": SIG})

def s_cpty(m, pr, prev_build, build_id):
    """Relationship churn. Writes state(m) and returns (aggregates per customer, moved dollars per customer × dir × bank)."""
    K = ["cust", "dir", "scope", "other"]
    cur = pr.where(F.col("scope") != "self").select(*K, F.col("amount").cast("double").alias("a"), F.col("o_fi").alias("fi"))
    J = (load_state("cpty", m, prev_build).join(cur.withColumn("_c", F.lit(1)), K, "full")
           .withColumn("h", F.coalesce("mask", F.lit(0)))
           .withColumn("c", F.col("_c").isNotNull())
           .withColumn("in_prev", F.expr("(h & 1) = 1"))
           .withColumn("seen", F.col("h") != 0)
           .withColumn("standing", F.expr(f"bit_count(h & {REC_MASK}) >= {REC_MIN}"))
           .withColumn("moved_now", F.col("c") & F.col("fi").isNotNull() & F.col("last_fi").isNotNull()
                                    & (F.col("fi") != F.col("last_fi")))
           .persist(MD))
    write_state(J.select(*K, F.expr(f"(shiftleft(h, 1) | cast(c as int)) & {FULL_MASK}").cast("int").alias("mask"),
                         F.when(F.col("c"), F.col("a")).otherwise(F.col("last_amt")).alias("last_amt"),
                         F.when(F.col("c"), F.coalesce("fi", "last_fi")).otherwise(F.col("last_fi")).alias("last_fi"))
                  .where("mask != 0"), "cpty", m, build_id)
    c = lambda s, d: (F.col("scope") == s) & (F.col("dir") == d)
    isd = lambda d: F.col("dir") == d
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0)).cast("long")
    sa = lambda cond, x: F.sum(F.when(cond, F.col(x)))
    ag = []
    for s in S2:
        for d in D:
            ag += [cnt(c(s, d) & F.col("c") & ~F.col("seen")).alias(f"new_cpty_{s}_{d}"),
                   cnt(c(s, d) & F.col("c") & F.col("seen") & ~F.col("in_prev")).alias(f"reactivated_cpty_{s}_{d}"),
                   cnt(c(s, d) & F.col("c") & F.col("in_prev")).alias(f"retained_cpty_{s}_{d}"),
                   cnt(c(s, d) & ~F.col("c") & F.col("in_prev")).alias(f"lost_cpty_{s}_{d}")]
    for d in D:
        ag += [cnt(isd(d) & F.col("in_prev")).alias(f"_prevn_{d}"),
               cnt(isd(d) & F.col("c") & F.col("in_prev")).alias(f"_retn_{d}"),
               sa(isd(d) & F.col("c") & ~F.col("seen"), "a").alias(f"_newa_{d}"),
               sa(isd(d) & F.col("in_prev"), "last_amt").alias(f"_preva_{d}"),
               sa(isd(d) & ~F.col("c") & F.col("in_prev"), "last_amt").alias(f"_losta_{d}"),
               cnt(isd(d) & F.col("standing")).alias(f"rec_standing_{d}"),
               cnt(isd(d) & F.col("standing") & ~F.col("c")).alias(f"rec_broken_{d}"),
               sa(isd(d) & F.col("standing"), "last_amt").alias(f"_stda_{d}"),
               sa(isd(d) & F.col("standing") & ~F.col("c"), "last_amt").alias(f"_brka_{d}"),
               cnt(c("ext", d) & F.col("in_prev") & F.col("moved_now")).alias(f"cpty_bank_moved_{d}")]
    ag.append(F.max((~F.col("c") & (F.col("in_prev") | F.col("standing"))).cast("int")).alias("_keep"))
    agg = J.groupBy("cust").agg(*ag)
    moved = (J.where((F.col("scope") == "ext") & F.col("last_fi").isNotNull())
              .groupBy("cust", "dir", F.col("last_fi").alias("fi"))
              .agg(F.sum(F.when(F.col("in_prev"), F.col("last_amt"))).alias("prev_lm_amt"),
                   F.sum(F.when(F.col("in_prev") & F.col("moved_now"), F.col("last_amt"))).alias("moved_lm_amt"),
                   F.sum("last_amt").alias("prev_all_amt"),
                   F.sum(F.when(F.col("moved_now"), F.col("last_amt"))).alias("moved_all_amt")))
    return J, agg, moved

def s_fi(m, l1, moved, prev_build, build_id):
    """Bank churn. Writes state(m) and returns customer × direction × bank rows (active, last month, or standing)."""
    K = ["cust", "dir", "fi"]
    cur = (l1.where((F.col("scope") == "ext") & F.col("fi_c").isNotNull())
             .groupBy("cust", "dir", F.col("fi_c").alias("fi"))
             .agg(F.sum("amount").cast(DEC).alias("amount"), F.sum("n_txn").cast("long").alias("n_txn"),
                  F.count("*").cast("long").alias("n_cpty"), F.sum("ntxn_ach").cast("long").alias("ntxn_ach"),
                  F.sum("amt_selfpay").alias("_self")))
    JF = (load_state("fi", m, prev_build).join(cur.withColumn("_c", F.lit(1)), K, "full")
            .withColumn("h", F.coalesce("mask", F.lit(0)))
            .withColumn("active", F.col("_c").isNotNull())
            .withColumn("in_prev", F.expr("(h & 1) = 1"))
            .withColumn("seen", F.col("h") != 0)
            .withColumn("standing", F.expr(f"bit_count(h & {REC_MASK}) >= {REC_MIN}"))
            .join(moved, K, "left")
            .withColumn("moved_lost", ~F.col("active") & F.col("in_prev") & (F.coalesce("prev_lm_amt", F.lit(0.0)) > 0)
                        & (F.coalesce("moved_lm_amt", F.lit(0.0)) >= F.lit(MOVE_SHARE) * F.col("prev_lm_amt")))
            .withColumn("moved_now", ~F.col("active") & (F.coalesce("prev_all_amt", F.lit(0.0)) > 0)
                        & (F.coalesce("moved_all_amt", F.lit(0.0)) >= F.lit(MOVE_SHARE) * F.col("prev_all_amt")))
            # sticky: once a bank's disappearance is explained by its counterparties moving, it stays explained for the
            # months it still counts as a broken standing bank, until the bank is active again
            .withColumn("moved_any", ~F.col("active") & (F.coalesce(F.col("moved"), F.lit(False)) | F.col("moved_lost") | F.col("moved_now")))
            .withColumn("moved_broken", F.col("standing") & F.col("moved_any"))
            .persist(MD))
    write_state(JF.select(*K, F.expr(f"(shiftleft(h, 1) | cast(active as int)) & {FULL_MASK}").cast("int").alias("mask"),
                          F.when(F.col("active"), F.col("amount").cast("double")).otherwise(F.col("last_amt")).alias("last_amt"),
                          F.col("moved_any").alias("moved"))
                  .where("mask != 0"), "fi", m, build_id)
    flags = (JF.where(F.col("active") | F.col("in_prev") | F.col("standing"))
               .select(*K, F.coalesce("amount", F.lit(0).cast(DEC)).cast(DEC).alias("amount"),
                       F.coalesce("n_txn", F.lit(0)).cast("long").alias("n_txn"),
                       F.coalesce("n_cpty", F.lit(0)).cast("long").alias("n_cpty"),
                       F.coalesce("ntxn_ach", F.lit(0)).cast("long").alias("ntxn_ach"),
                       F.round(F.coalesce("_self", F.lit(0.0)), 2).cast(DEC).alias("selfpay_amt"),
                       F.when(~F.col("active"), F.col("last_amt")).alias("last_amt"),
                       "active", "in_prev", "seen", "standing", "moved_lost", "moved_broken", "moved_any"))
    return JF, flags

def s_hub(m, pr):
    cpm = spark.read.parquet(month_in("cpm", m))
    dup = cpm.groupBy("ent_id").count().where("count > 1").limit(5).toPandas()
    if len(dup):
        raise RuntimeError(f"{T_CPM} {m}: ent_id not unique, e.g. {dup.ent_id.tolist()}")
    cpm = cpm.select(F.col("ent_id").alias("other"), F.coalesce(F.col("eligible").cast("boolean"), F.lit(False)).alias("elig"),
                     F.coalesce(F.col("hub_tier").cast("int"), F.lit(0)).alias("tier"), F.lit(1).alias("_hit"))
    x = (pr.where(F.col("scope") == "ext").select("cust", "dir", "other", F.col("amount").cast("double").alias("a"))
           .join(cpm, "other", "left"))
    isd = lambda d: F.col("dir") == d
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0)).cast("long")
    ag = []
    for d in D:
        ag += [cnt(isd(d) & F.col("elig")).alias(f"n_cpty_elig_{d}"),
               F.sum(F.when(isd(d) & F.col("elig"), F.col("a")).otherwise(0.0)).alias(f"_elig_{d}")]
        for k in HUB_TIERS:
            ag += [cnt(isd(d) & (F.col("tier") >= k)).alias(f"n_hub_t{k}_{d}"),
                   F.sum(F.when(isd(d) & (F.col("tier") >= k), F.col("a")).otherwise(0.0)).alias(f"_hub{k}_{d}")]
    ag += [F.sum(F.when(F.col("_hit").isNull(), F.col("a")).otherwise(0.0)).alias("_cpm_miss_amt"),
           F.sum(F.when(F.col("_hit").isNull(), 1).otherwise(0)).cast("long").alias("_cpm_miss_n")]
    return x.groupBy("cust").agg(*ag)

def s_sector(pr):
    nx = (pr.where((F.col("scope") == "int") & F.col("o_naics2").isNotNull())
            .select("cust", "dir", "o_naics2", F.col("amount").cast("double").alias("a"))
            .join(CUST.select(F.col("mdm_id").alias("cust"), F.col("naics2").alias("own_naics2")), "cust", "left"))
    isd = lambda d: F.col("dir") == d
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0)).cast("long")
    g = nx.groupBy("cust", "dir", "o_naics2").agg(F.sum("a").alias("a"), F.count("*").alias("n"))
    g = g.withColumn("p", F.col("a") / F.sum("a").over(Window.partitionBy("cust", "dir")))
    sect = g.groupBy("cust").agg(*[x for d in D for x in (
        cnt(isd(d)).alias(f"_nn_{d}"), F.sum(F.when(isd(d), F.col("n")).otherwise(0)).alias(f"_ncp_{d}"),
        F.max(F.when(isd(d), F.col("p"))).alias(f"_ntop_{d}"),
        F.sum(F.when(isd(d) & (F.col("p") > 0), -F.col("p") * F.log(F.col("p")))).alias(f"_nent_{d}"))])
    same = nx.where(F.col("own_naics2").isNotNull()).groupBy("cust").agg(*[x for d in D for x in (
        F.sum(F.when(isd(d) & (F.col("o_naics2") == F.col("own_naics2")), F.col("a")).otherwise(0.0)).alias(f"_same_{d}"),
        F.sum(F.when(isd(d), F.col("a"))).alias(f"_valid_{d}"))])
    return sect.join(same, "cust", "full")

def s_days(m):
    dd = spark.read.parquet(month_in("edge_d", m)).where(~F.col("is_self"))
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0)).cast("long")
    return (dd.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"), "txn_date")
              .unionByName(dd.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"), "txn_date"))
              .distinct().groupBy("cust").agg(*[cnt(F.col("dir") == d).alias(f"active_days_{d}") for d in D]))

In [ ]:
# §3b  Assembly: one row per customer, typed by the dictionary
def assemble(m, l1, pr, cagg, flags, hub, sect, days):
    amt = F.col("amount").cast("double")
    c = lambda s, d: (F.col("scope") == s) & (F.col("dir") == d)
    isd = lambda d: F.col("dir") == d
    cnt = lambda cond: F.sum(F.when(cond, 1).otherwise(0)).cast("long")

    # flows, rails and counterparty-type dollars (from l1)
    ag = []
    for s in S3:
        for d in D:
            ag += [F.sum(F.when(c(s, d), F.col("amount"))).alias(f"amt_{s}_{d}"),
                   F.sum(F.when(c(s, d), F.col("n_txn"))).alias(f"ntxn_{s}_{d}")]
    for d in D:
        ag += [F.sum(F.when(c("ext", d), F.col("amt_biz")).otherwise(0.0)).alias(f"_biz_{d}"),
               F.sum(F.when(c("ext", d), F.col("amt_person")).otherwise(0.0)).alias(f"_person_{d}"),
               F.sum(F.when(c("ext", d), F.col("amt_selfpay")).otherwise(0.0)).alias(f"_selfpay_{d}"),
               F.sum(F.when(c("int", d), F.col("amt_tm")).otherwise(0.0)).alias(f"_tm_{d}"),
               F.sum(F.when(c("int", d), F.col("amt_org")).otherwise(0.0)).alias(f"_org_{d}")]
        ag += [F.sum(F.when(isd(d) & (F.col("scope") != "self"), F.col(f"amt_{r}")).otherwise(0.0)).alias(f"_rail_{r}_{d}")
               for r in RAILS]
    flow = l1.groupBy("cust").agg(*ag)

    # breadth and concentration (pairs, self excluded)
    ns = pr.where(F.col("scope") != "self").withColumn("a", amt)
    ag = [cnt(c(s, d)).alias(f"ncpty_{s}_{d}") for s in S2 for d in D]
    for d in D:
        ag += [F.sum(F.when(isd(d), F.col("a"))).alias(f"_tot_{d}"), F.max(F.when(isd(d), F.col("a"))).alias(f"_max_{d}"),
               F.sum(F.when(isd(d), F.col("a") * F.col("a"))).alias(f"_sq_{d}")]
    conc = ns.groupBy("cust").agg(*ag)

    # banks (customer × direction × bank rows)
    act = F.col("active")
    a_fi = F.col("amount").cast("double")
    is_pnc = F.col("fi").isin(PNC_FI)
    ag = []
    for d in D:
        ag += [cnt(isd(d) & act).alias(f"n_fi_{d}"), cnt(isd(d) & act & (F.col("ntxn_ach") > 0)).alias(f"n_fi_ach_{d}"),
               F.max(F.when(isd(d) & act, a_fi)).alias(f"_fimax_{d}"), F.sum(F.when(isd(d) & act, a_fi)).alias(f"_fitot_{d}"),
               F.sum(F.when(isd(d) & act & is_pnc, a_fi).otherwise(0.0)).alias(f"_pnc_{d}"),
               F.max(F.when(isd(d) & act, F.struct(a_fi.alias("a"), F.col("fi")))).alias(f"_top_{d}"),
               cnt(isd(d) & act & (F.col("selfpay_amt") > 0)).alias(f"selfpay_n_fi_{d}"),
               F.max(F.when(isd(d) & act & (F.col("selfpay_amt") > 0),
                            F.struct(F.col("selfpay_amt").cast("double").alias("a"), F.col("fi")))).alias(f"_stop_{d}"),
               cnt(isd(d) & act & ~F.col("seen")).alias(f"new_fi_{d}"),
               cnt(isd(d) & act & F.col("seen") & ~F.col("in_prev")).alias(f"reactivated_fi_{d}"),
               cnt(isd(d) & act & F.col("in_prev")).alias(f"retained_fi_{d}"),
               cnt(isd(d) & ~act & F.col("in_prev")).alias(f"lost_fi_{d}"),
               cnt(isd(d) & F.col("moved_lost")).alias(f"lost_fi_cpty_moved_{d}"),
               cnt(isd(d) & F.col("standing")).alias(f"_fistd_{d}"),
               cnt(isd(d) & F.col("standing") & ~act).alias(f"fi_rec_broken_{d}"),
               cnt(isd(d) & F.col("moved_broken")).alias(f"fi_rec_broken_cpty_moved_{d}")]
    ag.append(F.max((~act & (F.col("in_prev") | F.col("standing"))).cast("int")).alias("_keep_fi"))
    banks = flags.groupBy("cust").agg(*ag)

    # base: active this month, or with a lost / broken relationship or bank (fully dark customers keep a row)
    base = (l1.select("cust").distinct()
              .unionByName(cagg.where(F.col("_keep") == 1).select("cust"))
              .unionByName(banks.where(F.col("_keep_fi") == 1).select("cust")).distinct())
    X = base
    for part in (flow, conc, banks.drop("_keep_fi"), cagg.drop("_keep"), hub, sect, days):
        X = X.join(part, "cust", "left")

    z = lambda x: F.coalesce(F.col(x).cast("double"), F.lit(0.0))
    sd = lambda num, den: F.when(den > 0, num / den)
    zero = lambda x, t=LNG: F.coalesce(F.col(x), F.lit(0)).cast(t)
    for s in S3:
        for d in D:
            X = X.withColumn(f"amt_{s}_{d}", zero(f"amt_{s}_{d}", DEC)).withColumn(f"ntxn_{s}_{d}", zero(f"ntxn_{s}_{d}"))
    for d in D:
        X = (X.withColumn(f"amt_all_{d}", (F.col(f"amt_int_{d}") + F.col(f"amt_ext_{d}")).cast(DEC))
              .withColumn(f"ntxn_all_{d}", F.col(f"ntxn_int_{d}") + F.col(f"ntxn_ext_{d}")))
        for s in S2:
            X = X.withColumn(f"ncpty_{s}_{d}", zero(f"ncpty_{s}_{d}"))
        X = X.withColumn(f"ncpty_all_{d}", F.col(f"ncpty_int_{d}") + F.col(f"ncpty_ext_{d}"))
        A, E, I = z(f"amt_all_{d}"), z(f"amt_ext_{d}"), z(f"amt_int_{d}")
        X = (X.withColumn(f"avg_ticket_{d}", sd(A, z(f"ntxn_all_{d}")))
              .withColumn(f"ext_share_{d}", sd(E, A))
              .withColumn(f"top1_share_{d}", sd(z(f"_max_{d}"), z(f"_tot_{d}")))
              .withColumn(f"hhi_{d}", sd(z(f"_sq_{d}"), z(f"_tot_{d}") * z(f"_tot_{d}")))
              .withColumn(f"top_fi_{d}", F.col(f"_top_{d}.fi"))
              .withColumn(f"top_fi_share_{d}", sd(z(f"_fimax_{d}"), z(f"_fitot_{d}")))
              .withColumn(f"fi_coverage_{d}", sd(z(f"_fitot_{d}"), E))
              .withColumn(f"pnc_fi_share_{d}", sd(z(f"_pnc_{d}"), z(f"_fitot_{d}")))
              .withColumn(f"selfpay_amt_{d}", F.round(z(f"_selfpay_{d}"), 2).cast(DEC))
              .withColumn(f"selfpay_share_{d}", sd(z(f"_selfpay_{d}"), E))
              .withColumn(f"selfpay_top_fi_{d}", F.col(f"_stop_{d}.fi"))
              .withColumn(f"ext_business_share_{d}", sd(z(f"_biz_{d}"), E))
              .withColumn(f"ext_person_share_{d}", sd(z(f"_person_{d}"), E))
              .withColumn(f"int_tm_share_{d}", sd(z(f"_tm_{d}"), I))
              .withColumn(f"int_org_share_{d}", sd(z(f"_org_{d}"), I)))
        guard = z(f"_ncp_{d}") >= MIN_NAICS_CP
        X = (X.withColumn(f"n_naics2_{d}", F.when(guard, F.col(f"_nn_{d}")))
              .withColumn(f"naics2_top_share_{d}", F.when(guard, F.col(f"_ntop_{d}")))
              .withColumn(f"naics2_entropy_{d}", F.when(guard, F.coalesce(F.col(f"_nent_{d}"), F.lit(0.0))))
              .withColumn(f"same_naics2_share_{d}", sd(z(f"_same_{d}"), z(f"_valid_{d}"))))
        for r in RAILS:
            X = X.withColumn(f"share_{r}_{d}", sd(z(f"_rail_{r}_{d}"), A))
        X = X.withColumn(f"n_rails_{d}", sum((z(f"_rail_{r}_{d}") > 0).cast("int") for r in RAILS))
        for x in [f"{st}_cpty_{s}_{d}" for st in ("new", "reactivated", "retained", "lost") for s in S2] + [
                  f"rec_standing_{d}", f"rec_broken_{d}", f"cpty_bank_moved_{d}", f"n_fi_{d}", f"n_fi_ach_{d}",
                  f"selfpay_n_fi_{d}", f"new_fi_{d}", f"reactivated_fi_{d}", f"retained_fi_{d}", f"lost_fi_{d}",
                  f"lost_fi_cpty_moved_{d}", f"fi_rec_broken_{d}", f"fi_rec_broken_cpty_moved_{d}", f"active_days_{d}",
                  f"n_cpty_elig_{d}"] + [f"n_hub_t{k}_{d}" for k in HUB_TIERS]:
            X = X.withColumn(x, zero(x))
        X = (X.withColumn(f"retention_{d}", sd(z(f"_retn_{d}"), z(f"_prevn_{d}")))
              .withColumn(f"new_amt_share_{d}", sd(z(f"_newa_{d}"), A))
              .withColumn(f"lost_amt_share_{d}", sd(z(f"_losta_{d}"), z(f"_preva_{d}")))
              .withColumn(f"rec_broken_share_{d}", sd(z(f"rec_broken_{d}"), z(f"rec_standing_{d}")))
              .withColumn(f"rec_broken_amt_share_{d}", sd(z(f"_brka_{d}"), z(f"_stda_{d}")))
              .withColumn(f"fi_retention_{d}", sd(z(f"retained_fi_{d}"), z(f"retained_fi_{d}") + z(f"lost_fi_{d}")))
              .withColumn(f"fi_rec_broken_share_{d}", sd(z(f"fi_rec_broken_{d}"), z(f"_fistd_{d}")))
              .withColumn(f"fi_rec_broken_net_{d}", F.col(f"fi_rec_broken_{d}") - F.col(f"fi_rec_broken_cpty_moved_{d}"))
              .withColumn(f"fi_rec_broken_net_share_{d}", sd(F.col(f"fi_rec_broken_net_{d}").cast("double"), z(f"_fistd_{d}")))
              .withColumn(f"elig_share_{d}", sd(z(f"_elig_{d}"), E)))
        for k in HUB_TIERS:
            X = X.withColumn(f"hub_t{k}_share_{d}", sd(z(f"_hub{k}_{d}"), E))
    X = (X.withColumn("net_flow_all", (F.col("amt_all_in") - F.col("amt_all_out")).cast(DEC))
          .withColumn("net_flow_ext", (F.col("amt_ext_in") - F.col("amt_ext_out")).cast(DEC))
          .withColumn("passthrough_ratio", sd(F.least(z("amt_all_in"), z("amt_all_out")), F.greatest(z("amt_all_in"), z("amt_all_out"))))
          .withColumn("history_months", F.lit(min(LOOKBACK, months_between(HISTORY_START, m)))))

    ident = spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("cust"), *[x for x in IDENTITY_COLS if x != "mdm_id"])
    X = (X.join(ident, "cust", "left").withColumnRenamed("cust", "mdm_id")
          .withColumn("in_neo4j_customer", F.coalesce(F.col("in_neo4j_customer").cast("boolean"), F.lit(False)))
          .withColumn("is_tm", F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)))
          .withColumn("pwr_shared", F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False))))
    return X.select(*IDENTITY_COLS, *[F.col(x).cast(METRIC_TYPES[x]).alias(x) for x in METRIC_COLS])

def cust_fi_rows(flags):
    who = spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("cust"), "customer_pwr_id", "master_pwr_id", "customer_name",
                                         F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("_tm"))
    x = flags.join(who, "cust", "left")
    if CUST_FI_TM_ONLY:
        x = x.where(F.col("_tm"))
    x = (x.withColumn("status", F.when(F.col("active") & ~F.col("seen"), "new")
                                 .when(F.col("active") & ~F.col("in_prev"), "reactivated")
                                 .when(F.col("active"), "retained")
                                 .when(F.col("in_prev"), "lost").otherwise("lapsed"))
          .withColumn("rec_broken", F.col("standing") & ~F.col("active"))
          .withColumn("cpty_moved", F.col("moved_any"))
          .withColumn("is_pnc", F.col("fi").isin(PNC_FI))
          .withColumnRenamed("cust", "mdm_id"))
    return x.select(*[F.col(c_).cast(t_).alias(c_) for c_, t_, _ in FI_SPEC])

In [ ]:
# §3c  Run: resume, stage, reconcile, publish
def ensure_table(key, schema):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in schema.fields if f.name != "month")
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {OUT[key]} ({cols}) PARTITIONED BY (month STRING) "
              f"STORED AS PARQUET LOCATION '{OUT_P[key]}'")

def add_partition(key, m):
    spark.sql(f"ALTER TABLE {OUT[key]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{month_dir(key, m)}'")

def reregister(key):
    """Drop + re-create the table definition from the newest month's schema and re-attach every month on disk."""
    months = list_months(OUT_P[key])
    if not months:
        return
    schema = spark.read.parquet(month_dir(key, months[-1])).schema
    spark.sql(f"DROP TABLE IF EXISTS {OUT[key]}")
    ensure_table(key, schema)
    for mm in months:
        add_partition(key, mm)
    spark.catalog.refreshTable(OUT[key])

def marker(m):
    return read_json(f"{month_dir('metrics', m)}/{MARKER}")

def edge_totals(m):
    r = edges(m).agg(*[F.sum(F.when(F.col(k) == "CUST", F.col(v))).alias(f"{d}_{v}")
                       for d, k in (("out", "src_kind"), ("in", "dst_kind")) for v in ("amount", "n_txn")]).first()
    return {k: (r[k] if r[k] is not None else 0) for k in r.asDict()}

def offenders(m, mt):
    """Per-customer comparison, only when the month does not reconcile."""
    em = edges(m)
    e = (em.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("mdm_id"), F.col("amount").alias("e_out"), F.lit(0).cast(DEC).alias("e_in"))
           .unionByName(em.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("mdm_id"), F.lit(0).cast(DEC).alias("e_out"), F.col("amount").alias("e_in")))
           .groupBy("mdm_id").agg(F.sum("e_out").alias("e_out"), F.sum("e_in").alias("e_in")))
    g = mt.select("mdm_id", (F.col("amt_int_out") + F.col("amt_ext_out") + F.col("amt_self_out")).alias("g_out"),
                  (F.col("amt_int_in") + F.col("amt_ext_in") + F.col("amt_self_in")).alias("g_in"))
    return (e.join(g, "mdm_id", "full").na.fill(0)
             .withColumn("diff", F.abs(F.col("e_out") - F.col("g_out")) + F.abs(F.col("e_in") - F.col("g_in")))
             .where("diff != 0").orderBy(F.desc("diff")).limit(20).toPandas())

def build_month(m, prev_build, log):
    fp = month_fp(m)
    wm = read_json(f"{WORK}/month={m}/_PKG_WORK.json")
    if not (wm and wm.get("sig") == SIG and wm.get("fp") == fp and wm.get("prev_build") == prev_build):
        rm(f"{WORK}/month={m}")
        wm = {"sig": SIG, "fp": fp, "prev_build": prev_build, "build_id": uuid.uuid4().hex[:12]}
        write_json(f"{WORK}/month={m}/_PKG_WORK.json", wm)
    build_id, t_month = wm["build_id"], time.time()
    done = lambda step: exists(f"{work_path(m, step)}/_SUCCESS")
    rd = lambda step: spark.read.parquet(work_path(m, step))

    def run(step, fn, n_files=None):
        t0 = time.time()
        if not done(step):
            df = fn()
            (df.repartition(n_files) if n_files else df).write.mode("overwrite").parquet(work_path(m, step))
        log["steps"][step] = round((time.time() - t0) / 60, 2)
        return rd(step)

    lk = run("lk", lambda: s_lk(m))
    map_miss = lk.where("map_miss").limit(MAP_MISS_MAX + 20).toPandas()
    if len(map_miss) > MAP_MISS_MAX:
        raise RuntimeError(f"{m}: {len(map_miss)}+ counterparty ids missing from {T_MAP}, e.g. {map_miss.cpty_id.head(10).tolist()}")
    l1 = run("l1", lambda: s_l1(m, lk))
    pr = run("pairs", lambda: s_pairs(l1).repartition(FILES["pairs"], "cust").sortWithinPartitions("cust", "dir", "other"))

    # relationship churn: state(m) + aggregates + moved dollars, all written before the step counts as done
    t0 = time.time()
    if not (done("cpty_agg") and done("cpty_moved") and (read_json(f"{state_path('cpty', m)}/_PKG_STATE.json") or {}).get("build_id") == build_id):
        rm(work_path(m, "cpty_agg")); rm(work_path(m, "cpty_moved"))
        J, agg, moved = s_cpty(m, pr, prev_build, build_id)
        moved.write.mode("overwrite").parquet(work_path(m, "cpty_moved"))
        agg.write.mode("overwrite").parquet(work_path(m, "cpty_agg"))
        J.unpersist()
    log["steps"]["cpty"] = round((time.time() - t0) / 60, 2)
    cagg, moved = rd("cpty_agg"), rd("cpty_moved")

    t0 = time.time()
    if not (done("fi_flags") and (read_json(f"{state_path('fi', m)}/_PKG_STATE.json") or {}).get("build_id") == build_id):
        rm(work_path(m, "fi_flags"))
        JF, flags = s_fi(m, l1, moved, prev_build, build_id)
        flags.write.mode("overwrite").parquet(work_path(m, "fi_flags"))
        JF.unpersist()
    log["steps"]["fi"] = round((time.time() - t0) / 60, 2)
    flags = rd("fi_flags")

    hub = run("hub", lambda: s_hub(m, pr))
    sect = run("sector", lambda: s_sector(pr))
    days = run("days", lambda: s_days(m))

    # stage the three outputs
    t0 = time.time()
    X = assemble(m, l1, pr, cagg, flags, hub, sect, days)
    X.repartition(FILES["metrics"], "mdm_id").sortWithinPartitions("mdm_id").write.mode("overwrite") \
     .option("compression", CODEC).parquet(stage_path("metrics", m))
    (cust_fi_rows(flags).repartition(FILES["cust_fi"], "mdm_id").sortWithinPartitions("mdm_id", "dir", "fi")
       .write.mode("overwrite").option("compression", CODEC).parquet(stage_path("cust_fi", m)))
    log["steps"]["assemble"] = round((time.time() - t0) / 60, 2)

    # reconcile exactly: metrics and pairs against the edges, customer × bank against l1
    t0 = time.time()
    exp = edge_totals(m)
    mt = spark.read.parquet(stage_path("metrics", m))
    g = mt.agg(*[F.sum(F.col(f"amt_int_{d}") + F.col(f"amt_ext_{d}") + F.col(f"amt_self_{d}")).alias(f"{d}_amount") for d in D],
               *[F.sum(F.col(f"ntxn_int_{d}") + F.col(f"ntxn_ext_{d}") + F.col(f"ntxn_self_{d}")).alias(f"{d}_n_txn") for d in D],
               F.sum(F.col("amt_ext_in") + F.col("amt_ext_out")).alias("ext_amount"),
               F.count("*").alias("rows"), F.countDistinct("mdm_id").alias("distinct"),
               F.sum(F.col("is_tm").cast("int")).alias("tm"), F.sum((~F.col("in_neo4j_customer")).cast("int")).alias("not_in_dim")).first()
    gp = pr.groupBy("dir").agg(F.sum("amount").alias("a"), F.sum("n_txn").alias("n")).collect()
    gp = {f"{r['dir']}_{k}": r[v] for r in gp for k, v in (("amount", "a"), ("n_txn", "n"))}
    tm_ids = CUST.where("is_tm").select(F.col("mdm_id").alias("cust")) if CUST_FI_TM_ONLY else None
    l1e = l1.where((F.col("scope") == "ext") & F.col("fi_c").isNotNull())
    if tm_ids is not None:
        l1e = l1e.join(tm_ids, "cust", "inner")
    efi = {r["dir"]: r["a"] for r in l1e.groupBy("dir").agg(F.sum("amount").alias("a")).collect()}
    gfi = {r["dir"]: r["a"] for r in spark.read.parquet(stage_path("cust_fi", m)).groupBy("dir").agg(F.sum("amount").alias("a")).collect()}
    hm = hub.agg(F.sum("_cpm_miss_amt").alias("a"), F.sum("_cpm_miss_n").alias("n")).first()
    ext_amt = float(g["ext_amount"] or 0)
    checks = {"metrics $ and txns = edges": all((g[k] or 0) == (exp[k] or 0) for k in exp),
              "pairs $ and txns = edges": all((gp.get(k) or 0) == (exp[k] or 0) for k in exp),
              "one row per customer": g["rows"] == g["distinct"],
              "customer × bank $ = l1": all((gfi.get(d) or 0) == (efi.get(d) or 0) for d in D),
              "entities found in cpty metrics": (hm["a"] or 0) <= CPM_MISS_MAX_SHARE * max(ext_amt, 1.0)}
    log["steps"]["reconcile"] = round((time.time() - t0) / 60, 2)
    log.update({"customers": g["rows"], "tm_customers": g["tm"], "not_in_cust_dim": g["not_in_dim"],
                "out_usd_bn": float(g["out_amount"] or 0) / 1e9, "in_usd_bn": float(g["in_amount"] or 0) / 1e9,
                "map_miss_ids": len(map_miss), "cpm_miss_cpty": int(hm["n"] or 0), "cpm_miss_usd": float(hm["a"] or 0)})
    failed = [k for k, ok in checks.items() if not ok]
    if failed:
        detail = offenders(m, mt) if not checks["metrics $ and txns = edges"] else pd.DataFrame()
        show(BAD(f"{m}: reconciliation failed: {failed}"),
             TB("Expected (edges) vs got", pd.DataFrame([{"total": k, "edges": exp[k], "metrics": g[k], "pairs": gp.get(k)} for k in exp])),
             TB("Customer × bank vs l1", pd.DataFrame([{"dir": d, "l1": efi.get(d), "cust_fi": gfi.get(d)} for d in D])),
             TB("Largest per-customer differences", detail) if len(detail) else None,
             N(f"pkg_cpty_metrics_monthly misses: {log['cpm_miss_cpty']:,} pairs, {log['cpm_miss_usd']:,.0f} USD"))
        raise AssertionError(f"{m}: reconciliation failed: {failed}. Nothing was published; the month will be redone.")

    # publish: outputs into place, then the marker (the marker is what makes the month done)
    for key in ("pairs", "cust_fi", "metrics"):
        mv(work_path(m, "pairs") if key == "pairs" else stage_path(key, m), month_dir(key, m))
        ensure_table(key, spark.read.parquet(month_dir(key, m)).schema)
        add_partition(key, m)
    write_json(f"{month_dir('metrics', m)}/{MARKER}",
               {"sig": SIG, "fp": fp, "build_id": build_id, "prev_build_id": prev_build, "version": METRICS_VERSION,
                "built_at": dt.datetime.now().isoformat(timespec="seconds"), "customers": g["rows"]})
    # prune: keep KEEP_STATES states, drop this month's work folder
    keep = {m}
    x = m
    for _ in range(KEEP_STATES - 1):
        x = prev_month(x); keep.add(x)
    for kind in ("cpty", "fi"):
        for mm in list_months(f"{STATE}/kind={kind}"):
            if mm < m and mm not in keep:
                rm(state_path(kind, mm))
    rm(f"{WORK}/month={m}")
    log["minutes"] = round((time.time() - t_month) / 60, 2)
    with open("pkg_cust_metrics_v3_runlog.csv", "a") as f:
        f.write(f"{dt.datetime.now().isoformat(timespec='seconds')},{m},{log['minutes']},{g['rows']},{SIG},{build_id}\n")
    return build_id

# ---- which months: available in every monthly input, then the done / stale chain ------------------------------------
AVAIL = sorted(set.intersection(*[set(list_months(IN_LOC[k])) for k in IN_LOC]))
last = LAST_MONTH or (AVAIL[-1] if AVAIL else None)
RUN_MONTHS = month_range(HISTORY_START, last)
missing = [m for m in RUN_MONTHS if m not in AVAIL]
if missing:
    raise RuntimeError(f"months missing from at least one monthly input: {missing}. "
                       f"Per input: {({k: [m for m in missing if m not in set(list_months(v))] for k, v in IN_LOC.items()})}")
def can_start_at(m):
    """A month can be (re)built only from HISTORY_START or when the previous month's published states are still kept."""
    if m == HISTORY_START:
        return True
    pmk = marker(prev_month(m))
    return bool(pmk) and all((read_json(f"{state_path(k, prev_month(m))}/_PKG_STATE.json") or {}).get("build_id")
                             == pmk.get("build_id") for k in ("cpty", "fi"))
if REBUILD_FROM:
    if REBUILD_FROM not in RUN_MONTHS:
        raise RuntimeError(f"REBUILD_FROM = {REBUILD_FROM} is outside {RUN_MONTHS[0]} → {RUN_MONTHS[-1]}")
    if not can_start_at(REBUILD_FROM):
        raise RuntimeError(f"REBUILD_FROM = {REBUILD_FROM} needs the churn state of {prev_month(REBUILD_FROM)}, which is no "
                           f"longer kept (KEEP_STATES = {KEEP_STATES}). Months that can be rebuilt from now: "
                           f"{[m for m in RUN_MONTHS if can_start_at(m)]}. Nothing was changed.")
    for m in [x for x in RUN_MONTHS if x >= REBUILD_FROM]:
        rm(f"{month_dir('metrics', m)}/{MARKER}")
    print(f"REBUILD_FROM = {REBUILD_FROM}: markers removed for {REBUILD_FROM} onward. Set REBUILD_FROM = None after this run.")

status, prev_mk, chain_ok = [], None, True
for m in RUN_MONTHS:
    mk = marker(m)
    ok = (chain_ok and mk is not None and mk.get("sig") == SIG and mk.get("fp") == month_fp(m)
          and exists(f"{month_dir('metrics', m)}/_SUCCESS")
          and (m == HISTORY_START or (prev_mk is not None and mk.get("prev_build_id") == prev_mk.get("build_id"))))
    why = "done" if ok else ("after a stale month" if not chain_ok else "no marker" if mk is None else
                             "settings or static inputs changed" if mk.get("sig") != SIG else
                             "monthly inputs changed" if mk.get("fp") != month_fp(m) else "previous month rebuilt")
    status.append({"month": m, "status": why})
    chain_ok = chain_ok and ok
    prev_mk = mk if ok else None
todo = [s["month"] for s in status if s["status"] != "done"]
if MAX_MONTHS_PER_RUN:
    todo = todo[:MAX_MONTHS_PER_RUN]
st_df = pd.DataFrame(status).groupby("status").month.agg(["count", "min", "max"]).reset_index()

if todo and not can_start_at(todo[0]):
    why0 = next(s["status"] for s in status if s["month"] == todo[0])
    raise RuntimeError(f"{todo[0]} has to be rebuilt ({why0}), but the churn state of {prev_month(todo[0])} is no longer kept "
                       f"(KEEP_STATES = {KEEP_STATES}). Set REBUILD_FROM = HISTORY_START to rebuild the chain. Nothing was changed.")
MONTH_LOG = []
handle = display(render(H(f"Signature {SIG} · {len(RUN_MONTHS)} months {RUN_MONTHS[0]} → {RUN_MONTHS[-1]} · to build now: {len(todo)}"),
                        TB("Month status", st_df)), display_id=True)
prev_build = None
if todo:
    pm = prev_month(todo[0])
    prev_build = None if todo[0] == HISTORY_START else (marker(pm) or {}).get("build_id")
for m in todo:
    log = {"month": m, "steps": {}}
    try:
        prev_build = build_month(m, prev_build, log)
    except Exception as e:
        log["error"] = str(e).splitlines()[0][:200]
        MONTH_LOG.append(log)
        handle.update(render(H(f"Stopped at {m}: rerun this cell to resume from {m}"), BAD(log["error"]),
                             TB("This run", pd.DataFrame([{k: v for k, v in r.items() if k != "steps"} for r in MONTH_LOG]))))
        raise
    MONTH_LOG.append(log)
    rows = pd.DataFrame([{**{k: v for k, v in r.items() if k != "steps"}, **{f"min_{k}": v for k, v in r["steps"].items()}}
                         for r in MONTH_LOG])
    handle.update(render(H(f"Signature {SIG} · built {len(MONTH_LOG)} of {len(todo)} this run · last {m}"),
                         TB("Month status before this run", st_df), TB("This run", rows)))
for key in ("pairs", "cust_fi", "metrics"):
    reregister(key)
handle.update(render(H(f"Signature {SIG} · built {len(MONTH_LOG)} of {len(todo)} this run · all reconciled"),
                     TB("Month status before this run", st_df),
                     TB("This run", pd.DataFrame([{**{k: v for k, v in r.items() if k != "steps"},
                                                   **{f"min_{k}": v for k, v in r["steps"].items()}} for r in MONTH_LOG]))
                     if MONTH_LOG else N("Nothing to build: every month is done for this signature."),
                     N("Table definitions re-created from the current schema: run INVALIDATE METADATA in Impala (§5).")))

## §4 QA on the latest month

- Distribution of the main metrics for TM customers.
- **v2 vs v3** on the same month, for the columns whose definition changed (entity level, canonical banks). This is the
  check to look at before pointing attrition at `_v3`: `n_fi_ach_out` is `fin_out_n`.
- **Bank moves:** how much of the lost and broken bank count is counterparty-driven, and which counterparties moved
  (their main bank changed while the customer kept paying them).

In [ ]:
# §4  QA
built = [m for m in RUN_MONTHS if marker(m)]
blocks = []
if built:
    lm = built[-1]
    mt = spark.read.parquet(month_dir("metrics", lm)).where(F.col("is_tm"))
    look = ["ncpty_all_out", "ncpty_ext_in", "n_fi_out", "n_fi_in", "n_fi_ach_out", "top_fi_share_in", "pnc_fi_share_out",
            "lost_fi_in", "lost_fi_cpty_moved_in", "fi_rec_broken_in", "fi_rec_broken_cpty_moved_in", "fi_rec_broken_net_share_in",
            "cpty_bank_moved_out", "rec_broken_share_out", "selfpay_share_out", "selfpay_n_fi_out", "hub_t1_share_out",
            "hub_t3_share_in", "elig_share_out", "ext_person_share_in", "passthrough_ratio"]
    qs = mt.select(*[F.expr(f"percentile_approx({c_}, array(0.1, 0.5, 0.9))").alias(f"q_{c_}") for c_ in look],
                   *[F.avg(F.col(c_).isNull().cast("double")).alias(f"n_{c_}") for c_ in look],
                   *[F.avg((F.col(c_) > 0).cast("double")).alias(f"p_{c_}") for c_ in look]).first()
    dist = pd.DataFrame([{"metric": c_, "p10": (qs[f"q_{c_}"] or [None] * 3)[0], "p50": (qs[f"q_{c_}"] or [None] * 3)[1],
                          "p90": (qs[f"q_{c_}"] or [None] * 3)[2], "null_share": qs[f"n_{c_}"], "nonzero_share": qs[f"p_{c_}"]}
                         for c_ in look])
    tot = mt.agg(*[F.sum(c_).alias(c_) for d in D for c_ in (f"lost_fi_{d}", f"lost_fi_cpty_moved_{d}", f"fi_rec_broken_{d}",
                                                              f"fi_rec_broken_cpty_moved_{d}", f"cpty_bank_moved_{d}")]).first()
    mv_tab = pd.DataFrame([{"direction": d, "lost banks": tot[f"lost_fi_{d}"], "lost: counterparty moved": tot[f"lost_fi_cpty_moved_{d}"],
                            "lost: moved share": (tot[f"lost_fi_cpty_moved_{d}"] or 0) / max(tot[f"lost_fi_{d}"] or 0, 1),
                            "broken standing banks": tot[f"fi_rec_broken_{d}"],
                            "broken: counterparty moved": tot[f"fi_rec_broken_cpty_moved_{d}"],
                            "broken: moved share": (tot[f"fi_rec_broken_cpty_moved_{d}"] or 0) / max(tot[f"fi_rec_broken_{d}"] or 0, 1),
                            "retained counterparties that changed bank": tot[f"cpty_bank_moved_{d}"]} for d in D])
    blocks += [H(f"QA · {lm} · TM customers"), TB("Distribution", dist, pct=("null_share", "nonzero_share")),
               TB("Lost and broken banks: how many are driven by the counterparty", mv_tab,
                  pct=("lost: moved share", "broken: moved share"))]
    pm = prev_month(lm)
    if pm in built:
        p0 = spark.read.parquet(month_dir("pairs", pm)).where("scope = 'ext' AND o_fi IS NOT NULL").select("cust", "dir", "other", F.col("o_fi").alias("fi_prev"))
        p1 = spark.read.parquet(month_dir("pairs", lm)).where("scope = 'ext' AND o_fi IS NOT NULL").select("cust", "dir", "other", F.col("o_fi").alias("fi_now"))
        moves = (p0.join(p1, ["cust", "dir", "other"]).where("fi_prev != fi_now")
                   .groupBy("other", "fi_prev", "fi_now").agg(F.countDistinct("cust").alias("customers"))
                   .orderBy(F.desc("customers")).limit(15)
                   .join(spark.read.parquet(month_in("cpm", lm)).select(F.col("ent_id").alias("other"), "ent_name"), "other", "left")
                   .select("ent_name", "fi_prev", "fi_now", "customers", "other").orderBy(F.desc("customers")).toPandas())
        blocks += [TB(f"Counterparties whose main bank changed {pm} → {lm} while customers kept paying them", moves)]
    v2m = table_location(T_V2_METRICS) if table_exists(T_V2_METRICS) else None
    common = [x for x in built if v2m and exists(f"{v2m}/month={x}")]
    if COMPARE_V2 and common:
        cm_ = common[-1]
        cmp_cols = ["ncpty_ext_out", "ncpty_ext_in", "n_fi_out", "n_fi_in", "n_fi_ach_out", "n_fi_ach_in", "lost_fi_in",
                    "lost_fi_out", "fi_rec_broken_in", "fi_rec_broken_out", "rec_broken_out", "new_cpty_ext_out",
                    "lost_cpty_ext_out", "selfpay_n_fi_out", "ext_business_share_out"]
        cmp_cols = [x for x in cmp_cols if x in set(spark.read.parquet(f"{v2m}/month={cm_}").columns)]   # v1-built months lack some
        v2 = spark.read.parquet(f"{v2m}/month={cm_}").where("is_tm").select("mdm_id", *[F.col(x).alias(f"v2_{x}") for x in cmp_cols])
        v3 = spark.read.parquet(month_dir("metrics", cm_)).where("is_tm").select("mdm_id", *[F.col(x).alias(f"v3_{x}") for x in cmp_cols])
        j = v2.join(v3, "mdm_id", "inner")
        r = j.agg(F.count("*").alias("n"),
                  *[F.avg(f"v2_{x}").alias(f"a2_{x}") for x in cmp_cols], *[F.avg(f"v3_{x}").alias(f"a3_{x}") for x in cmp_cols],
                  *[F.avg((F.abs(F.coalesce(F.col(f"v3_{x}"), F.lit(0)) - F.coalesce(F.col(f"v2_{x}"), F.lit(0)))
                           > 0.1 * F.greatest(F.abs(F.coalesce(F.col(f"v2_{x}"), F.lit(0))), F.lit(1e-9))).cast("double")).alias(f"c_{x}")
                    for x in cmp_cols],
                  *[F.corr(f"v2_{x}", f"v3_{x}").alias(f"r_{x}") for x in cmp_cols]).first()
        cmp = pd.DataFrame([{"metric": x, "v2 mean": r[f"a2_{x}"], "v3 mean": r[f"a3_{x}"],
                             "customers changed > 10%": r[f"c_{x}"], "correlation": r[f"r_{x}"]} for x in cmp_cols])
        blocks += [TB(f"v2 vs v3 · {cm_} · {r['n']:,} TM customers in both", cmp, pct=("customers changed > 10%",)),
                   N("Expected: banks and lost banks drop slightly (canonical names); external breadth and churn move for "
                     "a few percent of customers (ent_l2). A large change in n_fi_ach_out would need a look before attrition switches.")]
    # storage
    st = []
    for key in ("pairs", "cust_fi", "metrics"):
        ms = list_months(OUT_P[key])
        tot_b = sum(fs_for(month_dir(key, x)).getContentSummary(JPath(month_dir(key, x))).getLength() for x in ms)
        st.append({"table": OUT[key], "months": len(ms), "GB_total": tot_b / 1e9, "GB_per_month": tot_b / max(len(ms), 1) / 1e9})
    sb = fs_for(STATE).getContentSummary(JPath(STATE)).getLength() if exists(STATE) else 0
    st.append({"table": "churn state (working, latest months only)", "months": KEEP_STATES, "GB_total": sb / 1e9,
               "GB_per_month": sb / max(KEEP_STATES, 1) / 1e9})
    blocks += [TB("Storage (logical; HDFS keeps 3 copies)", pd.DataFrame(st))]
else:
    blocks += [N("No month built yet.")]
show(*blocks)

## §5 Impala

Run once per new or re-created table: `INVALIDATE METADATA`. After a later Spark run that only added months: `REFRESH`.

In [ ]:
# §5  Impala statements
show(TB("Impala", pd.DataFrame({"statement": [f"INVALIDATE METADATA {OUT[k]};" for k in ("metrics", "pairs", "cust_fi", "dict")]
                                              + [f"REFRESH {OUT[k]};" for k in ("metrics", "pairs", "cust_fi", "dict")]})))